# CIFAR-10 图像分类任务

**教学说明（本科生）：**

本节介绍图像分类任务，对比传统机器学习方法和卷积神经网络（CNN）的性能。

**CIFAR-10数据集：**
- 10个类别：飞机、汽车、鸟、猫、鹿、狗、青蛙、马、船、卡车
- 50000张训练图像，10000张测试图像
- 图像尺寸：32×32×3（RGB彩色图像）

**学习内容：**
1. 传统方法：像素展平 + Logistic回归
2. 卷积神经网络：CNN
3. 对比两种方法的性能和效率

**核心概念：**
- **图像张量**：PyTorch中图像格式为[C, H, W]
- **卷积层**：提取局部特征，保持空间结构
- **池化层**：降低特征图尺寸，减少参数
- **全连接层**：分类决策

**学习目标：**
- 理解图像分类的基本流程
- 对比传统方法（像素展平+LR）和深度学习方法（CNN）的性能差异
- 理解卷积核的视觉含义
- 掌握CNN的基本结构：卷积→池化→全连接

### 本实验的"传统 vs 深度学习"对比

| 方法 | CIFAR-10 | MNIST | 核心思想 |
|------|---------|-------|---------|
| 传统基线 | 像素展平+LR | 像素展平+LR | 把图像当一维向量 |
| 深度学习 | CNN | CNN | 保持二维结构，提取空间特征 |

通过对比可以发现：**CNN之所以强，是因为它利用图像的局部空间结构，而不是把像素打平成一维。**1. 理解图像数据在计算机中的表示方式
2. 体会CNN在图像任务中的优势
3. 掌握深度学习图像处理的基本流程

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- CNN的层级结构及其作用（卷积→池化→全连接）
- 卷积核、特征图的可视化含义
- 传统方法与深度学习的核心差异
- 分类任务与回归任务的异同
- 模型训练和评估的基本流程
- 如何通过可视化理解模型的内部行为
- 将理论知识与代码实现对应起来的能力


> 💡 **教学提示**：卷积核可视化展示了"模型在看什么"。第一层卷积核通常学习边缘和颜色检测器，这和人眼视觉皮层的V1区功能非常相似！

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tqdm import tqdm

# 设置随机种子，保证结果尽量可复现
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("当前使用设备:", device)
print("PyTorch版本:", torch.__version__)

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

## 下载并读取 CIFAR-10 数据集

CIFAR-10 包含 10 类彩色图像，每张图像大小为 `32 × 32 × 3`。

10 个类别分别是：

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`

### 学习重点

- 了解CIFAR-10数据集的下载和加载流程
- 查看数据集的基本信息（样本数、图像尺寸、通道数）
- 理解深度学习中的数据加载机制

In [ ]:
# CIFAR-10类别名称
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# 图像预处理：转为Tensor，并进行标准化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

# 为了课堂演示速度，这里只取部分数据
# 完整训练集50000张，测试集10000张
train_dataset_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset_full = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_subset_size = 8000
test_subset_size = 2000

train_dataset = Subset(train_dataset_full, range(train_subset_size))
test_dataset = Subset(test_dataset_full, range(test_subset_size))

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("完整训练集数量:", len(train_dataset_full))
print("完整测试集数量:", len(test_dataset_full))
print("课堂演示训练集数量:", len(train_dataset))
print("课堂演示测试集数量:", len(test_dataset))
print("类别数量:", len(class_names))
print("类别名称:", class_names)

## 检查一个 batch 的数据形状

重点观察：

- `images.shape = [batch_size, channels, height, width]`
- PyTorch 中图像张量通常是 `C × H × W`
- 常见图片显示格式是 `H × W × C`

### 学习重点

- 理解深度学习中的batch概念（一次性送入模型的样本数）
- 掌握查看张量形状的方法，这是调试模型的必备技能
- 图像数据形状为 [B, C, H, W]：B=batch, C=通道, H=高, W=宽

In [ ]:
images, labels = next(iter(train_loader))

print("一个batch的图像形状:", images.shape)
print("一个batch的标签形状:", labels.shape)
print("第一张图像的形状:", images[0].shape)
print("第一张图像的标签编号:", labels[0].item())
print("第一张图像的类别名称:", class_names[labels[0].item()])

print("\n图像张量数值范围：")
print("最小值:", images.min().item())
print("最大值:", images.max().item())
print("均值:", images.mean().item())
print("标准差:", images.std().item())

## 可视化图像样本

由于前面做了标准化，显示图像时需要反标准化。

### 学习重点

- 通过可视化直观了解数据的样貌
- 观察不同类别图像的差异和特点
- 理解数据探索（EDA）在机器学习项目中的重要性

In [ ]:
def unnormalize(img_tensor):
    """将标准化后的图像还原到接近原始显示范围。"""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img

def show_images(images, labels, n=16):
    plt.figure(figsize=(10, 10))
    for i in range(n):
        img = unnormalize(images[i])
        img = img.permute(1, 2, 0).numpy()

        plt.subplot(4, 4, i + 1)
        plt.imshow(img)
        plt.title(class_names[labels[i].item()])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_images(images, labels, n=16)

## 查看类别分布

课堂解释点：

如果类别分布严重不均衡，模型可能偏向样本多的类别。
CIFAR-10 本身是相对均衡的数据集。

### 学习重点

- 检查类别是否均衡——类别不平衡会影响模型训练
- CIFAR-10的10个类别样本数应该大致相等

In [ ]:
# 统计训练子集中每个类别的数量
train_labels = [train_dataset_full.targets[i] for i in range(train_subset_size)]
label_counts = pd.Series(train_labels).value_counts().sort_index()

label_df = pd.DataFrame({
    "类别编号": label_counts.index,
    "类别名称": [class_names[i] for i in label_counts.index],
    "样本数量": label_counts.values
})

display(label_df)

plt.figure(figsize=(10, 4))
plt.bar(label_df["类别名称"], label_df["样本数量"])
plt.xticks(rotation=45)
plt.xlabel("类别")
plt.ylabel("样本数量")
plt.title("训练子集类别分布")
plt.tight_layout()
plt.show()

## 传统方法 baseline：像素展平 + Logistic Regression

本环节的教学目标：让学生看到**传统机器学习处理图像的完整步骤**。

传统方法不直接理解“图像结构”，而是把图像当成普通表格数据处理：

1. 读取图像：原始张量为 `[3, 32, 32]`；
2. 展平图像：把 `3 × 32 × 32` 转成 `3072` 维向量；
3. 构造训练矩阵：`X = [样本数, 3072]`；
4. 构造标签向量：`y = [样本数]`；
5. 用 Logistic Regression 学习“像素值 → 图像类别”的映射；
6. 在测试集上预测并计算准确率。

该方法的核心问题：**图像的空间结构被破坏**。例如，相邻像素之间的局部关系不会被模型显式利用。

### 学习重点

- 理解"像素展平"的含义：将2D图像变成1D向量
- 对比这种简单方法的性能上限
- 为后续CNN的性能提升提供对比基准

In [ ]:
# 为了传统分类器运行更快，只取较小的数据量
baseline_train_size = 3000
baseline_test_size = 800

print("=" * 80)
print("传统方法 Step 1：准备把图像数据转换成普通机器学习表格数据")
print("=" * 80)
print("训练样本数量:", baseline_train_size)
print("测试样本数量:", baseline_test_size)
print("单张CIFAR-10图像原始形状: [channel, height, width] = [3, 32, 32]")
print("展平后特征维度: 3 × 32 × 32 =", 3 * 32 * 32)

def dataset_to_numpy(dataset_full, size, show_steps=True):
    X_list = []
    y_list = []

    for i in range(size):
        img, label = dataset_full[i]

        if show_steps and i == 0:
            print("\n" + "=" * 80)
            print("传统方法 Step 2：观察第1张图像展平前后的变化")
            print("=" * 80)
            print("第1张图像原始张量 shape:", img.shape)
            print("第1张图像标签编号:", label)
            print("第1张图像标签名称:", class_names[label])
            print("第1张图像前3个像素通道的部分数值:")
            print(img[:, :2, :5])

        # img: [3, 32, 32] -> [3072]
        img_flatten = img.view(-1).numpy()

        if show_steps and i == 0:
            print("\n第1张图像展平后 shape:", img_flatten.shape)
            print("展平后前20个特征值:")
            print(img_flatten[:20])
            print("\n说明：展平后，图像已经不再显式保留 32×32 的二维空间结构。")

        X_list.append(img_flatten)
        y_list.append(label)

    X = np.array(X_list)
    y = np.array(y_list)
    return X, y

X_train_base, y_train_base = dataset_to_numpy(train_dataset_full, baseline_train_size, show_steps=True)
X_test_base, y_test_base = dataset_to_numpy(test_dataset_full, baseline_test_size, show_steps=False)

print("\n" + "=" * 80)
print("传统方法 Step 3：形成机器学习训练矩阵")
print("=" * 80)
print("X_train_base shape:", X_train_base.shape)
print("y_train_base shape:", y_train_base.shape)
print("X_test_base shape:", X_test_base.shape)
print("y_test_base shape:", y_test_base.shape)
print("\n含义：")
print("X_train_base: [样本数, 特征数]，每一行是一张被展平的图像")
print("y_train_base: [样本数]，每个值是对应图像的类别编号")

# 打印前5个样本的标签
preview_df = pd.DataFrame({
    "样本序号": list(range(5)),
    "标签编号": y_train_base[:5],
    "类别名称": [class_names[i] for i in y_train_base[:5]]
})
display(preview_df)

In [ ]:
print("=" * 80)
print("传统方法 Step 4：训练 Logistic Regression 分类器")
print("=" * 80)

print("模型输入维度:", X_train_base.shape[1])
print("模型输出类别数:", len(class_names))

print("训练目标：")
print("学习 3072 个像素特征 与 10 个图像类别之间的关系")

print("\n开始训练 Logistic Regression ...")
print("训练过程中会输出迭代进度。\n")

baseline_model = LogisticRegression(
    max_iter=1000,
    solver="saga",
    multi_class="multinomial",
    n_jobs=-1,
    verbose=1   # 开启实时进度打印
)

baseline_model.fit(
    X_train_base,
    y_train_base
)

print("\n训练完成。")

print("\n" + "=" * 80)
print("传统方法 Step 5：在测试集上预测")
print("=" * 80)

y_pred_base = baseline_model.predict(X_test_base)

y_prob_base = baseline_model.predict_proba(X_test_base)

baseline_acc = accuracy_score(
    y_test_base,
    y_pred_base
)

print("预测结果 shape:", y_pred_base.shape)

print("预测概率矩阵 shape:", y_prob_base.shape)

print("含义:")
print("[测试样本数, 类别数]")

result_preview = pd.DataFrame({
    "样本序号": list(range(10)),
    
    "真实标签": y_test_base[:10],
    
    "真实类别": [
        class_names[i]
        for i in y_test_base[:10]
    ],
    
    "预测标签": y_pred_base[:10],
    
    "预测类别": [
        class_names[i]
        for i in y_pred_base[:10]
    ],
    
    "最大预测概率": y_prob_base[:10].max(axis=1)
})

display(result_preview)

print("\n传统像素分类器准确率:")
print(baseline_acc)

print("\n分类报告：")

print(classification_report(
    y_test_base,
    y_pred_base,
    target_names=class_names
))

print("\n课堂解释：")

print("传统方法可以完成分类，")
print("但它把图像展平成一维向量，弱化了局部空间结构。")

print("\n例如：")
print("猫的耳朵、眼睛、轮廓这些局部模式，")
print("不会像 CNN 那样被专门提取。")

In [ ]:
# =========================
# 传统方法：展示展平后的表格数据
# 前6行特征矩阵
# =========================

print("=" * 80)
print("传统方法 Step 3.5：观察展平后的机器学习表格数据")
print("=" * 80)

# 构建DataFrame
flatten_df = pd.DataFrame(
    X_train_base[:6]
)

# 添加标签列
flatten_df["label"] = y_train_base[:6]

# 添加类别名称列
flatten_df["class_name"] = [
    class_names[i]
    for i in y_train_base[:6]
]

print("DataFrame shape:")
print(flatten_df.shape)

print("\n前6行数据：")
display(flatten_df)

print("\n课堂解释：")
print("传统机器学习会把每张图像转换成一行表格数据。")
print("每一列对应一个像素特征。")
print("3072 = 3 × 32 × 32。")
print("这种方式适合传统ML模型，但会弱化图像的二维空间结构。")

## 定义简单 CNN 模型

### CNN 的明确任务目标

本实验中，CNN 的目标不是“生成图像”，也不是“识别图像中所有物体的位置”，而是完成一个标准的**图像分类任务**：

> 输入一张 `3 × 32 × 32` 的 CIFAR-10 彩色图像，输出它属于 10 个类别中每一类的得分，并选择得分最高的类别作为预测结果。

形式化表达：

```text
输入 X: [batch_size, 3, 32, 32]
输出 y_pred: [batch_size, 10]
学习目标: 让预测类别尽可能接近真实标签
损失函数: CrossEntropyLoss
```

CNN 要学习的是：

1. 从像素中提取局部特征，例如边缘、纹理、颜色变化；
2. 将低级局部特征逐步组合成更高级的图像模式；
3. 根据这些特征判断图像属于 airplane、cat、dog 等哪一类。

CNN 的基本结构：

- `Conv2d`：提取局部图像特征；
- `ReLU`：引入非线性；
- `MaxPool2d`：降低空间尺寸，保留主要响应；
- `Linear`：把提取出的特征映射到 10 个类别。

### CNN 结构详解

| 层 | 输入形状 | 输出形状 | 参数量 | 作用 |
|----|---------|---------|--------|------|
| Conv2d(3→32,3×3) | [B,3,32,32] | [B,32,32,32] | 896 | 提取低阶特征（边缘、颜色） |
| ReLU | [B,32,32,32] | [B,32,32,32] | 0 | 引入非线性 |
| MaxPool2d(2×2) | [B,32,32,32] | [B,32,16,16] | 0 | 降采样，提取主要特征 |
| Conv2d(32→64,3×3) | [B,32,16,16] | [B,64,16,16] | 18496 | 提取中阶特征（纹理、形状） |
| ReLU | ... | ... | 0 | 引入非线性 |
| MaxPool2d(2×2) | [B,64,16,16] | [B,64,8,8] | 0 | 进一步降采样 |
| Flatten | [B,64,8,8] | [B,4096] | 0 | 展开成一维 |
| Linear(4096→256) | [B,4096] | [B,256] | 1,048,832 | 全连接特征整合 |
| Linear(256→10) | [B,256] | [B,10] | 2,570 | 输出10类得分 |

> 💡 **教学提示**：可以把CNN想象成"漏斗"——从大尺寸、少通道的图像开始，逐步变成小尺寸、多通道的特征图，最后展开成一维向量进行分类。

In [ ]:
class SimpleCNN(nn.Module):
    """用于正式训练的CNN模型：不打印中间张量，避免训练时输出过多。"""
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [batch, 16, 16, 16]
        x = self.pool(F.relu(self.conv2(x)))  # [batch, 32, 8, 8]
        x = x.view(x.size(0), -1)             # [batch, 2048]
        x = F.relu(self.fc1(x))               # [batch, 128]
        x = self.fc2(x)                       # [batch, 10]
        return x


class DebugCNN(SimpleCNN):
    """用于课堂讲解的CNN模型：只跑一个batch，用来打印每一层张量shape。"""
    def forward(self, x):
        print("=" * 80)
        print("CNN张量流动过程：输入图像 -> 卷积 -> 激活 -> 池化 -> 展平 -> 分类")
        print("=" * 80)
        print("输入张量 shape:", x.shape)
        print("含义: [batch, channel, height, width]")

        x = self.conv1(x)
        print("\n经过 Conv1 后 shape:", x.shape)
        print("解释: 16个卷积核提取16组局部特征，空间尺寸保持 32×32")

        x = F.relu(x)
        print("经过 ReLU1 后 shape:", x.shape)
        print("解释: ReLU不改变张量形状，只改变数值分布")

        x = self.pool(x)
        print("经过 Pool1 后 shape:", x.shape)
        print("解释: 池化使空间尺寸减半，32×32 -> 16×16")

        x = self.conv2(x)
        print("\n经过 Conv2 后 shape:", x.shape)
        print("解释: 32个卷积核提取更高层特征")

        x = F.relu(x)
        print("经过 ReLU2 后 shape:", x.shape)

        x = self.pool(x)
        print("经过 Pool2 后 shape:", x.shape)
        print("解释: 空间尺寸再次减半，16×16 -> 8×8")

        x = x.view(x.size(0), -1)
        print("\n展平 Flatten 后 shape:", x.shape)
        print("解释: 每张图像变为 32 × 8 × 8 = 2048 维特征向量")

        x = self.fc1(x)
        print("经过 FC1 后 shape:", x.shape)
        print("解释: 2048维特征被压缩到128维")

        x = F.relu(x)

        x = self.fc2(x)
        print("最终输出 shape:", x.shape)
        print("解释: 每张图像输出10个类别得分")

        return x

# 正式训练统一使用变量名 model，避免后续 cell 出现 NameError
model = SimpleCNN().to(device)
print(model)

# 统计模型参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n模型总参数量:", total_params)
print("可训练参数量:", trainable_params)
print("\nCNN训练目标: 最小化 CrossEntropyLoss，使模型输出的10类得分尽可能对应真实类别。")

In [ ]:
# 用 DebugCNN 跑一个 batch，只用于打印张量变化，不用于正式训练
# 注意：这里使用 debug_model，后续正式训练仍然使用 model = SimpleCNN()

debug_model = DebugCNN().to(device)

sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    debug_outputs = debug_model(sample_images)

print("\nDebugCNN输出示例:")
print("debug_outputs shape:", debug_outputs.shape)
print("第1张图像的10类得分:")
print(debug_outputs[0].cpu().numpy())

## 用一个 batch 检查 CNN 前向传播

重点：

- 输入是 `[64, 3, 32, 32]`
- 输出是 `[64, 10]`
- 每一行代表一张图像属于 10 个类别的得分

### 张量形状追踪

检查前向传播时，关注每个层的输入输出形状是否匹配：
```
输入图像: [B, 3, 32, 32]  ← CIFAR-10 的32×32彩色图
  ↓ Conv+Pool
特征图:   [B, 64, 8, 8]    ← 经过两次卷积+池化
  ↓ Flatten
一维向量: [B, 4096]         ← 展开
  ↓ Linear+Linear
类别得分: [B, 10]           ← 10个类别的logits
```

### 学习重点

- 理解深度学习中的batch概念（一次性送入模型的样本数）
- 掌握查看张量形状的方法，这是调试模型的必备技能
- 图像数据形状为 [B, C, H, W]：B=batch, C=通道, H=高, W=宽

In [ ]:
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_outputs = model(sample_images)

print("输入图像 batch 形状:", sample_images.shape)
print("模型输出形状:", sample_outputs.shape)
print("第一张图像的10类输出得分:")
print(sample_outputs[0].cpu().numpy())

predicted_class = torch.argmax(sample_outputs[0]).item()
true_class = sample_labels[0].item()

print("\n第一张图像真实类别编号:", true_class)
print("第一张图像真实类别名称:", class_names[true_class])
print("第一张图像预测类别编号:", predicted_class)
print("第一张图像预测类别名称:", class_names[predicted_class])
print("\n说明：模型训练前是随机初始化的，因此此时预测结果通常不可靠。")

## 训练 CNN 模型

训练阶段的明确目标：

> 通过反向传播不断更新卷积层和全连接层的参数，使 `CrossEntropyLoss` 逐渐降低，使测试集分类准确率逐渐提高。

默认训练 5 个 epoch，适合 90 min 课堂演示。

- 电脑较慢：可改成 3 个 epoch；
- 想获得更好结果：可改成 10–20 个 epoch；
- 课堂重点不是追求最高准确率，而是理解 CNN 如何从图像张量中学习分类特征。

> 💡 **教学提示**：CNN训练比全连接网络慢很多，因为卷积层需要大量计算。观察训练过程中损失是否持续下降，准确率是否持续上升。如果训练损失下降但验证损失上升，说明发生了过拟合。

In [ ]:
print("=" * 80)
print("CNN训练配置")
print("=" * 80)

print("任务类型: 10分类图像分类")
print("输入张量: [batch_size, 3, 32, 32]")
print("输出张量: [batch_size, 10]")
print("损失函数: CrossEntropyLoss")
print("优化器: Adam")
print("学习率: 0.001")
print("早停机制: 验证集连续5个epoch不提升则停止")

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 30

# =========================
# Early Stopping 配置
# =========================

patience = 5

best_test_acc = 0.0

epochs_no_improve = 0

best_model_state = None

# =========================
# 训练记录
# =========================

train_losses = []
train_accuracies = []
test_accuracies = []

# =========================
# 测试函数
# =========================

def evaluate(model, data_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

    return correct / total

# =========================
# 开始训练
# =========================

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    correct = 0
    total = 0

    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * labels.size(0)

        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

        loop.set_postfix(
            loss=loss.item()
        )

    # =========================
    # epoch统计
    # =========================

    epoch_loss = running_loss / total

    train_acc = correct / total

    test_acc = evaluate(
        model,
        test_loader
    )

    train_losses.append(epoch_loss)

    train_accuracies.append(train_acc)

    test_accuracies.append(test_acc)

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    print(f"训练损失: {epoch_loss:.4f}")

    print(f"训练准确率: {train_acc:.4f}")

    print(f"测试准确率: {test_acc:.4f}")

    # =========================
    # Early Stopping
    # =========================

    if test_acc > best_test_acc:

        best_test_acc = test_acc

        epochs_no_improve = 0

        best_model_state = model.state_dict()

        print("测试准确率提升，保存当前最佳模型。")

    else:

        epochs_no_improve += 1

        print(f"测试准确率未提升，累计 {epochs_no_improve} 次。")

    print("-" * 50)

    # =========================
    # 提前停止
    # =========================

    if epochs_no_improve >= patience:

        print("\n" + "=" * 80)
        print("触发 Early Stopping")
        print(f"连续 {patience} 个 epoch 测试准确率未提升")
        print("训练提前结束")
        print("=" * 80)

        break

# =========================
# 恢复最佳模型
# =========================

if best_model_state is not None:

    model.load_state_dict(best_model_state)

    print("\n已恢复最佳模型参数。")

# =========================
# 保存模型
# =========================

import os

save_dir = "./saved_models_cifar"

os.makedirs(save_dir, exist_ok=True)

model_save_path = os.path.join(
    save_dir,
    "cifar10_simple_cnn.pth"
)

checkpoint = {

    "model_state_dict": model.state_dict(),

    "class_names": class_names,

    "mean": (0.4914, 0.4822, 0.4465),

    "std": (0.2470, 0.2435, 0.2616),

    "input_size": (32, 32),

    "num_classes": 10,

    "model_name": "SimpleCNN",

    "best_test_accuracy": best_test_acc,

    "train_losses": train_losses,

    "train_accuracies": train_accuracies,

    "test_accuracies": test_accuracies,
}

torch.save(
    checkpoint,
    model_save_path
)

print("\n" + "=" * 80)
print("模型已保存")
print("=" * 80)

print("保存路径:", model_save_path)

print("最佳测试准确率:", best_test_acc)

## 可视化训练过程

观察点：

- 训练损失是否下降；
- 训练准确率是否上升；
- 测试准确率是否同步上升；
- 如果训练准确率远高于测试准确率，可能出现过拟合。

In [ ]:
actual_epochs = len(train_losses)

history_df = pd.DataFrame({
    "epoch": list(range(1, actual_epochs + 1)),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "test_accuracy": test_accuracies
})

display(history_df)

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN训练损失变化")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], marker="o", label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN准确率变化")
plt.legend()
plt.grid(True)
plt.show()

## CNN 测试集整体结果

输出：

- 测试准确率；
- 分类报告；
- 混淆矩阵。

### 学习重点

- 评估模型在**未见过的数据**上的表现
- 测试集准确率是衡量模型泛化能力的最终标准

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

cnn_acc = accuracy_score(all_labels, all_preds)

print("传统像素分类器准确率:", baseline_acc)
print("CNN测试集准确率:", cnn_acc)

print("\nCNN分类报告：")
print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

cm_df = pd.DataFrame(
    cm,
    index=[f"真实-{name}" for name in class_names],
    columns=[f"预测-{name}" for name in class_names]
)

display(cm_df)

plt.figure(figsize=(9, 7))
plt.imshow(cm, interpolation="nearest")
plt.title("CNN混淆矩阵")
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

plt.xlabel("预测类别")
plt.ylabel("真实类别")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.show()

## 可视化错误分类样本

这部分非常适合课堂讨论：

- 哪些类别容易混淆？
- 错误分类是否符合人类直觉？
- 模型是不是只学到了颜色、背景等浅层特征？

In [ ]:
# 收集错误分类样本
wrong_images = []
wrong_true = []
wrong_pred = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images_device = images.to(device)
        outputs = model(images_device)
        preds = torch.argmax(outputs, dim=1).cpu()

        wrong_mask = preds != labels

        for img, true_label, pred_label in zip(images[wrong_mask], labels[wrong_mask], preds[wrong_mask]):
            wrong_images.append(img)
            wrong_true.append(true_label.item())
            wrong_pred.append(pred_label.item())

        if len(wrong_images) >= 16:
            break

print("收集到的错误分类样本数量:", len(wrong_images))

plt.figure(figsize=(6, 6))
for i in range(min(16, len(wrong_images))):
    img = unnormalize(wrong_images[i])
    img = img.permute(1, 2, 0).numpy()

    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.title(
        f"True: {class_names[wrong_true[i]]}\nPred: {class_names[wrong_pred[i]]}",
        fontsize=10
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 可视化卷积核

这里展示第一层卷积核的形状和部分卷积核。

注意：第一层卷积核直接作用于 RGB 图像，所以可以作为小图像显示。

### 如何解读卷积核

卷积核（滤波器）可以理解为"特征检测器"：
- **第一层卷积核**：学习边缘、颜色、纹理等低级特征
- **中间层卷积核**：组合低级特征为形状、图案
- **深层卷积核**：识别物体部件（眼睛、轮子等）

这体现了深度学习的**层级特征学习**——从简单到复杂，从具体到抽象。

In [ ]:
conv1_weights = model.conv1.weight.data.cpu()

print("第一层卷积核权重形状:", conv1_weights.shape)
print("含义: [卷积核数量, 输入通道数, 卷积核高度, 卷积核宽度]")

plt.figure(figsize=(8, 4))
for i in range(min(16, conv1_weights.shape[0])):
    kernel = conv1_weights[i]

    # 为了显示，将数值缩放到0-1
    kernel_min = kernel.min()
    kernel_max = kernel.max()
    kernel_img = (kernel - kernel_min) / (kernel_max - kernel_min + 1e-8)

    kernel_img = kernel_img.permute(1, 2, 0).numpy()

    plt.subplot(4, 4, i + 1)
    plt.imshow(kernel_img)
    plt.title(f"Kernel {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 传统方法 *vs* CNN

课堂讨论重点：

1. 传统方法把图像展平成向量，破坏了空间结构；
2. CNN 保留了局部空间结构；
3. 卷积核可以学习边缘、纹理、颜色组合等局部模式；
4. 深层网络可以进一步组合低级特征，形成更高级的视觉语义。

In [ ]:
compare_df = pd.DataFrame({
    "方法": ["像素展平 + Logistic Regression", "简单 CNN"],
    "输入形式": ["3072维向量", "3×32×32图像张量"],
    "是否保留空间结构": ["否", "是"],
    "测试准确率": [baseline_acc, cnn_acc]
})

display(compare_df)

plt.figure(figsize=(7, 5))
plt.bar(compare_df["方法"], compare_df["测试准确率"])
plt.ylabel("Accuracy")
plt.title("传统方法与CNN分类准确率对比")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 模型应用

### 学习重点

- 将训练好的模型应用于新的、未见过数据
- 理解训练→验证→测试→应用的完整流程

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
import pandas as pd


# =========================
# 1. 定义 CNN 结构
# 必须与训练时一致
# =========================

class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            3,
            16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(
            32 * 8 * 8,
            128
        )

        self.fc2 = nn.Linear(
            128,
            10
        )

    def forward(self, x):

        x = self.pool(
            F.relu(self.conv1(x))
        )

        x = self.pool(
            F.relu(self.conv2(x))
        )

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x


# =========================
# 2. 读取模型文件
# =========================

model_path = "/Users/linzuhong/Desktop/teach/saved_models_cifar/cifar10_simple_cnn.pth" # 修改为你的模型权重路径

checkpoint = torch.load(
    model_path,
    map_location="cpu"
)

print("=" * 80)
print("模型文件读取成功")
print("=" * 80)

print("checkpoint keys:")
print(checkpoint.keys())


# =========================
# 3. 恢复模型
# =========================

model = SimpleCNN()

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("\n模型参数恢复成功")


# =========================
# 4. 读取类别名称和Normalize参数
# =========================

class_names = checkpoint["class_names"]

mean = checkpoint["mean"]

std = checkpoint["std"]

print("\n类别名称:")
print(class_names)

print("\nNormalize参数:")
print("mean =", mean)
print("std  =", std)


# =========================
# 5. 图像预处理
# =========================

transform = transforms.Compose([

    transforms.Resize((32, 32)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])


# =========================
# 6. 读取图片
# 修改为你的图片路径
# =========================

image_path = "/Users/linzuhong/Desktop/teach/saved_models_cifar/test/lihua.png" # 修改为你的图片路径

image = Image.open(image_path).convert("RGB")

print("\n原始图片大小:")
print(image.size)


# =========================
# 7. 显示原始图片
# =========================

plt.figure(figsize=(5,5))

plt.imshow(image)

plt.title("Original Image")

plt.axis("off")

plt.show()


# =========================
# 8. 图像转Tensor
# =========================

input_tensor = transform(image)

print("=" * 80)
print("图像Tensor信息")
print("=" * 80)

print("Tensor shape:")
print(input_tensor.shape)

print("\nTensor数值范围:")
print("最小值:", input_tensor.min().item())
print("最大值:", input_tensor.max().item())

print("\n前3个通道部分像素:")
print(input_tensor[:, :2, :5])


# =========================
# 9. 增加batch维度
# =========================

input_batch = input_tensor.unsqueeze(0)

print("\n加入batch后的shape:")
print(input_batch.shape)


# =========================
# 10. 模型预测
# =========================

with torch.no_grad():

    outputs = model(input_batch)

    probabilities = torch.softmax(
        outputs,
        dim=1
    )

    predicted_idx = torch.argmax(
        probabilities,
        dim=1
    ).item()

print("=" * 80)
print("模型输出")
print("=" * 80)

print("logits shape:")
print(outputs.shape)

print("\n10类 logits:")
print(outputs[0].numpy())

print("\n10类 probabilities:")
print(probabilities[0].numpy())

print("\n概率总和:")
print(probabilities.sum().item())


# =========================
# 11. 最终预测结果
# =========================

predicted_class = class_names[predicted_idx]

confidence = probabilities[0][predicted_idx].item()

print("=" * 80)
print("最终预测结果")
print("=" * 80)

print("预测类别编号:", predicted_idx)

print("预测类别名称:", predicted_class)

print("预测置信度:", confidence)


# =========================
# 12. 概率表格
# =========================

result_df = pd.DataFrame({

    "类别编号": list(range(len(class_names))),

    "类别名称": class_names,

    "预测概率": probabilities[0].numpy()
})

result_df = result_df.sort_values(
    by="预测概率",
    ascending=False
)

print("\n各类别预测概率:")

display(result_df)


# =========================
# 13. 概率柱状图
# =========================

plt.figure(figsize=(10,5))

plt.bar(
    result_df["类别名称"],
    result_df["预测概率"]
)

plt.xticks(rotation=45)

plt.ylabel("Probability")

plt.title("CNN Prediction Probabilities")

plt.grid(True)

plt.show()

# MNIST 手写数字识别任务

## 任务目标

本实验以 MNIST 手写数字数据集为对象，完成图像分类任务。

输入：一张 `28 × 28` 的灰度手写数字图像  
输出：预测该图像属于 `0–9` 中的哪一个数字类别

## 实验目标

1. 理解图像在计算机中的张量表示；
2. 掌握 MNIST 数据集的读取、打印和可视化；
3. 使用传统机器学习方法完成图像分类；
4. 构建 CNN 模型完成手写数字识别；
5. 打印 CNN 前向传播过程中的张量形状；
6. 对比传统方法和 CNN 的分类效果；
7. 分析混淆矩阵和错误分类样本。

### MNIST vs CIFAR-10 对比

| 特性 | MNIST | CIFAR-10 |
|------|-------|---------|
| 图像尺寸 | 28×28 | 32×32 |
| 通道数 | 1（灰度） | 3（彩色） |
| 类别数 | 10（数字） | 10（物体） |
| 难度 | 简单（二值化灰度图） | 困难（自然彩色图） |
| 传统方法性能 | ~92% | ~40% |
| CNN性能 | ~99% | ~75% |

### 学习重点

- 通过可视化直观了解数据的样貌
- 观察不同类别图像的差异和特点
- 理解数据探索（EDA）在机器学习项目中的重要性

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tqdm import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("当前使用设备:", device)
print("PyTorch版本:", torch.__version__)

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# CNN 几何回归任务：Fashion-MNIST 图像旋转角度预测

**教学说明（本科生）：**

本实验使用Fashion-MNIST数据集构建CNN回归模型。

**任务特点：**
- **回归任务**：预测连续的旋转角度值
- **几何理解**：CNN需要理解图像的方向信息

**学习内容：**
1. 回归任务与分类任务的区别
2. CNN用于回归的输出层设计
3. 旋转不变性学习

**学习目标：**
1. 理解CNN在回归任务中的应用
2. 掌握回归模型的评估方法
3. 体会空间信息对几何理解的重要性

### 分类 vs 回归

| 任务类型 | 输出 | 损失函数 | 评估指标 |
|---------|------|---------|---------|
| 分类 | 离散类别 | 交叉熵 | 准确率 |
| 回归 | 连续数值 | MSE/MAE | R², MAE |

本节的旋转角度预测是**回归任务**，CNN输出层不需要Softmax，而是直接输出一个数值。

---
### 学习重点

通过本节学习，你应该掌握：
- 理解上述概念的原理和意义
- CNN的层级结构及其作用（卷积→池化→全连接）
- 卷积核、特征图的可视化含义
- 分类任务与回归任务的异同
- 如何通过可视化理解模型的内部行为
- 将理论知识与代码实现对应起来的能力


> 💡 **教学提示**：卷积核可视化展示了"模型在看什么"。第一层卷积核通常学习边缘和颜色检测器，这和人眼视觉皮层的V1区功能非常相似！

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset_full = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_size = 10000
test_size = 2000

train_dataset = Subset(train_dataset_full, range(train_size))
test_dataset = Subset(test_dataset_full, range(test_size))

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("MNIST完整训练集数量:", len(train_dataset_full))
print("MNIST完整测试集数量:", len(test_dataset_full))
print("本实验使用训练集数量:", len(train_dataset))
print("本实验使用测试集数量:", len(test_dataset))

## 打印一个 batch 的张量信息

In [ ]:
images, labels = next(iter(train_loader))

print("一个 batch 的图像张量形状:", images.shape)
print("一个 batch 的标签张量形状:", labels.shape)

print("\n第一张图像的张量形状:", images[0].shape)
print("第一张图像的标签:", labels[0].item())

print("\n张量含义:")
print("batch_size =", images.shape[0])
print("channel =", images.shape[1])
print("height =", images.shape[2])
print("width =", images.shape[3])

print("\n图像张量数值统计:")
print("最小值:", images.min().item())
print("最大值:", images.max().item())
print("均值:", images.mean().item())
print("标准差:", images.std().item())

## 可视化 MNIST 图像

In [ ]:
def show_mnist_images(images, labels, n=16):
    plt.figure(figsize=(8, 8))
    
    for i in range(n):
        img = images[i].squeeze().numpy()
        
        plt.subplot(4, 4, i + 1)
        plt.imshow(img, cmap="gray")
        plt.title(f"Label: {labels[i].item()}")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

show_mnist_images(images, labels, n=16)

## 查看类别分布

### 学习重点

- 检查类别是否均衡——类别不平衡会影响模型训练
- CIFAR-10的10个类别样本数应该大致相等

In [ ]:
train_labels = [train_dataset_full.targets[i].item() for i in range(train_size)]

label_counts = pd.Series(train_labels).value_counts().sort_index()

label_df = pd.DataFrame({
    "数字类别": label_counts.index,
    "样本数量": label_counts.values
})

display(label_df)

plt.figure(figsize=(8, 4))
plt.bar(label_df["数字类别"].astype(str), label_df["样本数量"])
plt.xlabel("数字类别")
plt.ylabel("样本数量")
plt.title("MNIST训练子集类别分布")
plt.tight_layout()
plt.show()

## 传统机器学习方法：像素展平 + Logistic Regression

传统方法的基本流程：

1. 原始图像形状：`1 × 28 × 28`

2. 展平成一维向量：`784`

3. 把每张图片看作一个 784 维特征样本

4. 使用 Logistic Regression 完成 10 分类

传统方法的局限：

- 只把图像看成一串数字；

- 没有显式利用图像的空间结构；

- 不知道哪些像素是相邻的；

- 难以自动提取局部形状特征。

## 传统方法数据转换与打印

In [ ]:
baseline_train_size = 5000
baseline_test_size = 1000

def dataset_to_numpy(dataset, size):
    X_list = []
    y_list = []
    
    for i in range(size):
        img, label = dataset[i]
        
        if i == 0:
            print("第1张图像原始张量形状:", img.shape)
            print("第1张图像原始标签:", label)
            print("展平前元素总数:", img.numel())
        
        img_flat = img.view(-1).numpy()
        
        if i == 0:
            print("第1张图像展平后形状:", img_flat.shape)
            print("展平后的前20个数值:")
            print(img_flat[:20])
        
        X_list.append(img_flat)
        y_list.append(label)
    
    X = np.array(X_list)
    y = np.array(y_list)
    
    return X, y

X_train_base, y_train_base = dataset_to_numpy(train_dataset_full, baseline_train_size)
X_test_base, y_test_base = dataset_to_numpy(test_dataset_full, baseline_test_size)

print("\n传统方法训练数据矩阵形状:", X_train_base.shape)
print("传统方法测试数据矩阵形状:", X_test_base.shape)
print("训练标签形状:", y_train_base.shape)
print("测试标签形状:", y_test_base.shape)

print("\n解释:")
print("X_train_base.shape = [样本数量, 每张图像展平后的像素特征数]")
print("每张 MNIST 图像为 28 × 28 = 784 个像素")

## 展示展平后的MNIST表格数据

In [ ]:
print("=" * 80)
print("传统方法：MNIST 图像展平后的表格数据")
print("=" * 80)

# 构建DataFrame
mnist_flatten_df = pd.DataFrame(
    X_train_base[:6]
)

# 添加标签列
mnist_flatten_df["label"] = y_train_base[:6]

print("DataFrame shape:")
print(mnist_flatten_df.shape)

print("\n前6行数据：")
display(mnist_flatten_df)

print("\n课堂解释：")
print("传统机器学习会把每张 28×28 图像转换成一行表格数据。")
print("28 × 28 = 784，因此每张图像对应784个像素特征。")
print("DataFrame中的每一列，本质上对应一个像素位置。")
print("这种方式适合Logistic Regression等传统ML模型。")
print("但图像原本的二维空间结构会被展平。")

## 训练传统分类器

In [ ]:
print("=" * 80)
print("传统方法：Logistic Regression 手写数字分类")
print("=" * 80)

print("任务目标:")
print("学习 784 个像素特征 与 10 个数字类别之间的关系")

print("\n输入特征维度:", X_train_base.shape[1])

print("输出类别数量:", 10)

print("\n开始训练 Logistic Regression 分类器...")
print("训练过程中会实时输出优化进度。\n")

baseline_model = LogisticRegression(

    max_iter=1000,

    solver="saga",

    multi_class="multinomial",

    n_jobs=-1,

    verbose=1   # 开启训练进度打印
)

baseline_model.fit(
    X_train_base,
    y_train_base
)

print("\n训练完成。")

print("\n" + "=" * 80)
print("模型参数信息")
print("=" * 80)

print("模型参数矩阵形状:", baseline_model.coef_.shape)

print("模型截距形状:", baseline_model.intercept_.shape)

print("\n解释:")

print("coef_.shape = [类别数量, 输入特征数量]")

print("这里是 10 个数字类别，")
print("每个类别对应 784 个像素权重。")

print("\n即：")

print("每个数字类别，")
print("模型都会学习：")

print("哪些像素位置更重要。")

## 传统方法结果打印

In [ ]:
y_pred_base = baseline_model.predict(X_test_base)
y_prob_base = baseline_model.predict_proba(X_test_base)

baseline_acc = accuracy_score(y_test_base, y_pred_base)

print("传统方法测试准确率:", baseline_acc)

print("\n前10个真实标签:")
print(y_test_base[:10])

print("\n前10个预测标签:")
print(y_pred_base[:10])

print("\n前1张测试图像属于10个类别的预测概率:")
print(y_prob_base[0])

print("\n前1张测试图像预测类别:", y_pred_base[0])
print("前1张测试图像真实类别:", y_test_base[0])

In [ ]:
print("传统方法分类报告:")
print(classification_report(y_test_base, y_pred_base))

## CNN 模型任务目标

CNN 的目标是：

给定一张 MNIST 图像：

输入张量: [batch_size, 1, 28, 28]

模型输出每张图像属于 10 个数字类别的得分：

输出张量: [batch_size, 10]

其中 10 个输出值分别对应数字：

0, 1, 2, 3, 4, 5, 6, 7, 8, 9

训练目标：

通过反向传播不断调整 CNN 参数，使模型预测类别尽可能接近真实标签。

损失函数：`CrossEntropyLoss`

优化目标：

- 最小化分类错误
- 提高测试集准确率

> 💡 **教学提示**：CNN训练是最耗时的环节。观察训练过程中的loss下降趋势，判断模型是否正常收敛。如果loss震荡剧烈，可以尝试降低学习率。

## 定义 CNN 模型

In [ ]:
class SimpleMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=16,
            kernel_size=3,
            padding=1
        )
        
        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )
        
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  
        x = self.pool(F.relu(self.conv2(x)))  
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleMNISTCNN().to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n模型总参数量:", total_params)
print("可训练参数量:", trainable_params)

## CNN 张量打印

In [ ]:
class DebugMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        print("=" * 60)
        print("输入张量:")
        print(x.shape)
        print("含义: [batch_size, channel, height, width]")
        
        x = self.conv1(x)
        print("\n经过 Conv1 后:")
        print(x.shape)
        print("解释: 1个灰度通道变成16个特征图，图像尺寸保持28×28")
        
        x = F.relu(x)
        print("\n经过 ReLU 后:")
        print(x.shape)
        print("解释: ReLU不改变张量形状，只改变数值")
        
        x = self.pool(x)
        print("\n经过 Pool1 后:")
        print(x.shape)
        print("解释: 28×28 变成 14×14")
        
        x = self.conv2(x)
        print("\n经过 Conv2 后:")
        print(x.shape)
        print("解释: 16个特征图变成32个特征图，尺寸保持14×14")
        
        x = F.relu(x)
        x = self.pool(x)
        print("\n经过 Pool2 后:")
        print(x.shape)
        print("解释: 14×14 变成 7×7")
        
        x = x.view(x.size(0), -1)
        print("\n展平 Flatten 后:")
        print(x.shape)
        print("解释: 32 × 7 × 7 = 1568")
        
        x = self.fc1(x)
        print("\n经过 FC1 后:")
        print(x.shape)
        print("解释: 1568维特征压缩为128维")
        
        x = F.relu(x)
        x = self.fc2(x)
        print("\n最终输出:")
        print(x.shape)
        print("解释: 每张图像输出10个类别得分")
        print("=" * 60)
        
        return x

debug_model = DebugMNISTCNN().to(device)

sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    debug_outputs = debug_model(sample_images)

print("\n真实标签前10个:")
print(sample_labels[:10].numpy())

print("\n模型输出前1个样本的10类得分:")
print(debug_outputs[0].cpu().numpy())

print("\n预测类别:")
print(torch.argmax(debug_outputs[0]).item())

## 训练 CNN

> 💡 **教学提示**：CNN训练是最耗时的环节。观察训练过程中的loss下降趋势，判断模型是否正常收敛。如果loss震荡剧烈，可以尝试降低学习率。

In [ ]:
print("=" * 80)
print("CNN 手写数字识别训练")
print("=" * 80)

print("任务类型: MNIST 10分类")
print("输入张量: [batch_size, 1, 28, 28]")
print("输出张量: [batch_size, 10]")
print("损失函数: CrossEntropyLoss")
print("优化器: Adam")
print("学习率: 0.001")
print("Early Stopping: 连续5个epoch不提升则停止")

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 30

# =========================
# Early Stopping
# =========================

patience = 5
best_test_acc = 0.0
epochs_no_improve = 0
best_model_state = None

# =========================
# 训练记录
# =========================

train_losses = []
train_accuracies = []
test_accuracies = []

# =========================
# 测试函数
# =========================

def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = torch.argmax(
                outputs,
                dim=1
            )
            correct += (
                preds == labels
            ).sum().item()
            total += labels.size(0)
    return correct / total

# =========================
# 开始训练
# =========================

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(
            outputs,
            labels
        )
        loss.backward()
        optimizer.step()
        running_loss += (
            loss.item() * labels.size(0)
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        loop.set_postfix(
            loss=loss.item()
        )

    # =========================
    # epoch统计
    # =========================

    epoch_loss = running_loss / total
    train_acc = correct / total
    test_acc = evaluate(
        model,
        test_loader
    )
    train_losses.append(epoch_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
    print(f"训练损失: {epoch_loss:.4f}")
    print(f"训练准确率: {train_acc:.4f}")
    print(f"测试准确率: {test_acc:.4f}")

    # =========================
    # Early Stopping
    # =========================

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        epochs_no_improve = 0
        best_model_state = model.state_dict()
        print("测试准确率提升，保存当前最佳模型。")
    else:
        epochs_no_improve += 1
        print(f"测试准确率未提升，累计 {epochs_no_improve} 次。")
    print("-" * 50)

    # =========================
    # 提前停止
    # =========================

    if epochs_no_improve >= patience:

        print("\n" + "=" * 80)
        print("触发 Early Stopping")
        print(f"连续 {patience} 个 epoch 测试准确率未提升")
        print("训练提前结束")
        print("=" * 80)

        break

# =========================
# 恢复最佳模型
# =========================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    print("\n已恢复最佳模型参数。")

# =========================
# 保存模型
# =========================

import os

save_dir = "./saved_models_mnist"

os.makedirs(
    save_dir,
    exist_ok=True
)

model_save_path = os.path.join(
    save_dir,
    "mnist_simple_cnn.pth"
)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": [str(i) for i in range(10)],
    "input_size": (28, 28),
    "num_classes": 10,
    "model_name": "SimpleCNN",
    "best_test_accuracy": best_test_acc,
    "train_losses": train_losses,
    "train_accuracies": train_accuracies,
    "test_accuracies": test_accuracies,
}

torch.save(
    checkpoint,
    model_save_path
)

print("\n" + "=" * 80)
print("MNIST模型已保存")
print("=" * 80)

print("保存路径:", model_save_path)
print("最佳测试准确率:", best_test_acc)

## 训练过程可视化

In [ ]:
actual_epochs = len(train_losses)

history_df = pd.DataFrame({
    "epoch": range(1, actual_epochs + 1),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "test_accuracy": test_accuracies
})

display(history_df)

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN训练损失变化")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], marker="o", label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN准确率变化")
plt.legend()
plt.grid(True)
plt.show()

## CNN 测试结果

### 学习重点

- 评估模型在**未见过的数据**上的表现
- 测试集准确率是衡量模型泛化能力的最终标准

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

cnn_acc = accuracy_score(all_labels, all_preds)

print("传统方法准确率:", baseline_acc)
print("CNN准确率:", cnn_acc)

print("\nCNN分类报告:")
print(classification_report(all_labels, all_preds))

## 混淆矩阵

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

cm_df = pd.DataFrame(
    cm,
    index=[f"真实{i}" for i in range(10)],
    columns=[f"预测{i}" for i in range(10)]
)

display(cm_df)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("CNN混淆矩阵")
plt.colorbar()

plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))

plt.xlabel("预测类别")
plt.ylabel("真实类别")

for i in range(10):
    for j in range(10):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.show()

## 错误分类样本可视化

In [ ]:
wrong_images = []
wrong_true = []
wrong_pred = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images_device = images.to(device)
        outputs = model(images_device)
        preds = torch.argmax(outputs, dim=1).cpu()
        
        wrong_mask = preds != labels
        
        for img, true_label, pred_label in zip(images[wrong_mask], labels[wrong_mask], preds[wrong_mask]):
            wrong_images.append(img)
            wrong_true.append(true_label.item())
            wrong_pred.append(pred_label.item())
        
        if len(wrong_images) >= 16:
            break

print("收集到的错误分类样本数量:", len(wrong_images))

plt.figure(figsize=(8, 8))

for i in range(min(16, len(wrong_images))):
    img = wrong_images[i].squeeze().numpy()
    
    plt.subplot(4, 4, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"True: {wrong_true[i]}, Pred: {wrong_pred[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 可视化卷积核

### 如何解读卷积核

卷积核（滤波器）可以理解为"特征检测器"：
- **第一层卷积核**：学习边缘、颜色、纹理等低级特征
- **中间层卷积核**：组合低级特征为形状、图案
- **深层卷积核**：识别物体部件（眼睛、轮子等）

这体现了深度学习的**层级特征学习**——从简单到复杂，从具体到抽象。

In [ ]:
conv1_weights = model.conv1.weight.data.cpu()

print("=" * 80)
print("第一层卷积核信息")
print("=" * 80)

print("conv1 权重张量 shape:")
print(conv1_weights.shape)

print("\n含义:")
print("[卷积核数量, 输入通道数, 卷积核高度, 卷积核宽度]")
print("对于 MNIST：输入通道数 = 1")
print("所以每个卷积核可以看作一个 3×3 的灰度小图")

num_kernels = conv1_weights.shape[0]

plt.figure(figsize=(10, 6))

for i in range(num_kernels):
    kernel = conv1_weights[i, 0]

    # 归一化到 0~1，便于显示
    kernel_min = kernel.min()
    kernel_max = kernel.max()
    kernel_img = (kernel - kernel_min) / (kernel_max - kernel_min + 1e-8)

    plt.subplot(4, 4, i + 1)
    plt.imshow(kernel_img.numpy(), cmap="gray")
    plt.title(f"Kernel {i}")
    plt.axis("off")

plt.suptitle("CNN第一层卷积核可视化", fontsize=16)
plt.tight_layout()
plt.show()

## 传统方法 *vs* CNN 对比

In [ ]:
compare_df = pd.DataFrame({
    "方法": [
        "像素展平 + Logistic Regression",
        "CNN卷积神经网络"
    ],
    "输入形式": [
        "784维向量",
        "1×28×28图像张量"
    ],
    "是否保留空间结构": [
        "否",
        "是"
    ],
    "测试准确率": [
        baseline_acc,
        cnn_acc
    ]
})

display(compare_df)

plt.figure(figsize=(8, 5))
plt.bar(compare_df["方法"], compare_df["测试准确率"])
plt.ylabel("Accuracy")
plt.title("传统方法与CNN准确率对比")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 模型应用

### 学习重点

- 将训练好的模型应用于新的、未见过数据
- 理解训练→验证→测试→应用的完整流程

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from PIL import Image

import matplotlib.pyplot as plt
import pandas as pd


# =========================
# 定义模型
# =========================

class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(
            F.relu(self.conv1(x))
        )
        x = self.pool(
            F.relu(self.conv2(x))
        )
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


# =========================
# 加载模型
# =========================

checkpoint = torch.load(
    "/Users/linzuhong/Desktop/teach/saved_models_mnist/mnist_simple_cnn.pth", # 修改为你的模型权重路径
    map_location="cpu"
)

model = SimpleCNN()

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("模型读取成功")


# =========================
# 图像预处理
# =========================

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.1307,),
        std=(0.3081,)
    )
])


# =========================
# 读取图片
# =========================

image_path = "/Users/linzuhong/Desktop/teach/saved_models_mnist/test/7.png" # 修改为你的图片路径

image = Image.open(image_path)
plt.figure(figsize=(4,4))
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")
plt.show()


# =========================
# 转Tensor
# =========================

input_tensor = transform(image)

print("=" * 80)
print("Tensor 信息")
print("=" * 80)

print("shape:")
print(input_tensor.shape)

print("\n数值范围:")
print("min =", input_tensor.min().item())
print("max =", input_tensor.max().item())


# =========================
# 增加batch维度
# =========================

input_batch = input_tensor.unsqueeze(0)

print("\n加入batch后:")
print(input_batch.shape)


# =========================
# 模型预测
# =========================

with torch.no_grad():
    outputs = model(input_batch)
    probabilities = torch.softmax(
        outputs,
        dim=1
    )[0]
    pred_idx = torch.argmax(
        probabilities
    ).item()

print("=" * 80)
print("模型输出")
print("=" * 80)

print("logits:")
print(outputs[0].numpy())

print("\nprobabilities:")
print(probabilities.numpy())

print("\n预测结果:")
print("预测数字 =", pred_idx)

print("预测概率 =", probabilities[pred_idx].item())


# =========================
# 概率表格
# =========================

result_df = pd.DataFrame({
    "数字": list(range(10)),
    "概率": probabilities.numpy()
})

display(result_df)


# =========================
# 概率柱状图
# =========================

plt.figure(figsize=(8,4))
plt.bar(
    result_df["数字"].astype(str),
    result_df["概率"]
)

plt.xlabel("Digit")
plt.ylabel("Probability")
plt.title("Prediction Probabilities")
plt.grid(True)
plt.show()

# CNN 几何回归任务：Fashion-MNIST 图像旋转角度预测

## 任务目标

本实验使用 Fashion-MNIST 数据集构建 CNN 回归模型。

输入：
一张被随机旋转后的衣物图像

输出：
预测该图像的旋转角度（连续值）

例如：

```text
真实角度: +23.5°
预测角度: +21.8°

### 分类 vs 回归

| 任务类型 | 输出 | 损失函数 | 评估指标 |
|---------|------|---------|---------|
| 分类 | 离散类别 | 交叉熵 | 准确率 |
| 回归 | 连续数值 | MSE/MAE | R², MAE |

本节的旋转角度预测是**回归任务**，CNN输出层不需要Softmax，而是直接输出一个数值。

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tqdm import tqdm

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("当前设备:", device)
print("PyTorch版本:", torch.__version__)


plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

## 下载 Fashion-MNIST

In [ ]:
base_transform = transforms.ToTensor()

fashion_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=base_transform
)

print("Fashion-MNIST 样本数量:", len(fashion_dataset))

class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

print("\n类别名称:")
for i, name in enumerate(class_names):
    print(i, ":", name)

## 查看原始图像

In [ ]:
images = []
labels = []

for i in range(16):
    img, label = fashion_dataset[i]
    
    images.append(img)
    labels.append(label)

plt.figure(figsize=(8, 8))

for i in range(16):
    img = images[i].squeeze().numpy()
    
    plt.subplot(4, 4, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(class_names[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 构建“旋转角度预测”数据集

In [ ]:
class RotatedFashionMNIST(Dataset):
    
    def __init__(self, base_dataset, angle_range=(-45, 45)):
        self.base_dataset = base_dataset
        self.angle_range = angle_range
        
        self.normalize = transforms.Normalize(
            mean=(0.5,),
            std=(0.5,)
        )
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        
        image, label = self.base_dataset[idx]
        
        # tensor -> PIL
        pil_image = transforms.ToPILImage()(image)
        
        # 随机生成旋转角度
        angle = random.uniform(
            self.angle_range[0],
            self.angle_range[1]
        )
        
        rotated_image = pil_image.rotate(angle)
        
        rotated_tensor = transforms.ToTensor()(rotated_image)
        
        rotated_tensor = self.normalize(rotated_tensor)
        
        angle_tensor = torch.tensor(
            angle,
            dtype=torch.float32
        )
        
        return rotated_tensor, angle_tensor

rot_dataset = RotatedFashionMNIST(fashion_dataset)

print("旋转数据集样本数量:", len(rot_dataset))

## 打印张量信息

In [ ]:
sample_image, sample_angle = rot_dataset[0]

print("图像张量 shape:")
print(sample_image.shape)

print("\n张量含义:")
print("channel =", sample_image.shape[0])
print("height =", sample_image.shape[1])
print("width =", sample_image.shape[2])

print("\n真实旋转角度:")
print(sample_angle.item())

print("\n张量数值统计:")
print("最小值:", sample_image.min().item())
print("最大值:", sample_image.max().item())
print("均值:", sample_image.mean().item())
print("标准差:", sample_image.std().item())

## 可视化旋转图像

In [ ]:
plt.figure(figsize=(10, 10))

for i in range(16):
    
    image, angle = rot_dataset[i]
    
    image = image.squeeze().numpy()
    
    plt.subplot(4, 4, i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"{angle.item():.1f}°")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 数据集基本信息

### 📚 学习重点

- 记录数据集的基本统计信息
- 为模型设计提供依据

In [ ]:
max_samples = 12000

dataset_small, _ = random_split(
    rot_dataset,
    [max_samples, len(rot_dataset) - max_samples],
    generator=torch.Generator().manual_seed(seed)
)

train_size = int(0.8 * len(dataset_small))
test_size = len(dataset_small) - train_size

train_dataset, test_dataset = random_split(
    dataset_small,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(seed)
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("训练集数量:", len(train_dataset))
print("测试集数量:", len(test_dataset))

¥# 传统方法：图像展平 + Ridge 回归

传统方法流程：

1. 原始图像：`1 × 28 × 28`
2. 展平为：`784维向量`
3. 使用 Ridge 回归预测旋转角度

传统方法问题：

- 图像空间结构被破坏；
- 无法有效理解“方向”；
- 对旋转几何特征学习能力弱。

## 传统方法数据变换

In [ ]:
baseline_train_size = 3000
baseline_test_size = 800

def dataset_to_numpy(dataset, size):
    
    X_list = []
    y_list = []
    
    for i in range(size):
        
        image, angle = dataset[i]
        
        if i == 0:
            print("原始图像 shape:")
            print(image.shape)
            
            print("\n真实角度:")
            print(angle.item())
        
        image_flat = image.view(-1).numpy()
        
        if i == 0:
            print("\n展平后 shape:")
            print(image_flat.shape)
            
            print("\n前20个数值:")
            print(image_flat[:20])
        
        X_list.append(image_flat)
        y_list.append(angle.item())
    
    return np.array(X_list), np.array(y_list)

X_train_base, y_train_base = dataset_to_numpy(
    train_dataset,
    baseline_train_size
)

X_test_base, y_test_base = dataset_to_numpy(
    test_dataset,
    baseline_test_size
)

print("\n训练矩阵 shape:", X_train_base.shape)
print("测试矩阵 shape:", X_test_base.shape)

## 可视化展平后的表格数据

In [ ]:
print("=" * 80)
print("传统方法：Fashion-MNIST 图像展平后的回归数据")
print("=" * 80)

# 构建DataFrame
fashion_flatten_df = pd.DataFrame(
    X_train_base[:6]
)

# 添加旋转角标签
fashion_flatten_df["rotation_angle"] = y_train_base[:6]

print("DataFrame shape:")
print(fashion_flatten_df.shape)

print("\n前6行数据：")
display(fashion_flatten_df)

print("\n课堂解释：")
print("传统机器学习会把每张 Fashion-MNIST 图像展平成一行表格数据。")
print("28 × 28 = 784，因此每张图像对应784个像素特征。")
print("最后一列是该图像对应的旋转角度标签。")

print("\n此任务已经不是分类任务，而是回归任务。")

print("\n模型目标：")
print("根据图像像素特征，预测该服饰图像被旋转了多少度。")

print("\n传统方法中的输入输出：")
print("输入 X : 784维像素特征")
print("输出 y : 连续数值角度（例如 -23.5°）")

## 训练Ridge回归

In [ ]:
print("=" * 80)
print("传统方法：Ridge 回归进行旋转角预测")
print("=" * 80)

print("任务类型: 图像回归任务")
print("输入特征: 784维展平像素")
print("输出目标: 连续旋转角度")
print("模型: Ridge Regression")
print("正则化系数 alpha = 1.0")

print("\n训练目标:")
print("学习图像像素特征 与 图像旋转角度之间的关系")

print("\n训练集 shape:")
print(X_train_base.shape)

print("测试集 shape:")
print(X_test_base.shape)

print("\n开始训练 Ridge 回归模型...\n")


# =========================
# Ridge 回归
# =========================

baseline_model = Ridge(
    alpha=1.0
)

baseline_model.fit(
    X_train_base,
    y_train_base
)

print("训练完成。")


# =========================
# 模型参数分析
# =========================

print("\n" + "=" * 80)
print("模型参数信息")
print("=" * 80)

print("coef_ shape:")
print(baseline_model.coef_.shape)

print("\nintercept_:")
print(baseline_model.intercept_)

print("\n解释:")
print("coef_ 对应每个像素位置的权重")
print("模型会学习：")
print("哪些像素模式与旋转角度相关")


# =========================
# 预测
# =========================

print("\n" + "=" * 80)
print("开始预测测试集")
print("=" * 80)

y_pred_base = baseline_model.predict(
    X_test_base
)

print("预测完成。")

print("\n预测结果 shape:")
print(y_pred_base.shape)


# =========================
# 回归指标
# =========================

baseline_mae = mean_absolute_error(
    y_test_base,
    y_pred_base
)

baseline_mse = mean_squared_error(
    y_test_base,
    y_pred_base
)

baseline_rmse = np.sqrt(
    baseline_mse
)

baseline_r2 = r2_score(
    y_test_base,
    y_pred_base
)

print("\n" + "=" * 80)
print("Ridge 回归结果")
print("=" * 80)

print("MAE :", baseline_mae)

print("MSE :", baseline_mse)

print("RMSE:", baseline_rmse)

print("R²  :", baseline_r2)


# =========================
# 前10个结果
# =========================

result_df = pd.DataFrame({

    "真实角度": y_test_base[:10],

    "预测角度": y_pred_base[:10],

    "绝对误差": np.abs(
        y_test_base[:10] - y_pred_base[:10]
    )
})

print("\n前10个预测结果:")

display(result_df)


# =========================
# 课堂解释
# =========================

print("\n课堂解释:")

print("该任务已经不是分类任务，而是回归任务。")

print("\n模型不再输出:")
print("cat / dog / shirt")

print("\n而是输出:")
print("一个连续数值角度")

print("\n例如:")
print("-23.5°")
print("17.8°")
print("42.1°")

print("\nRidge Regression 本质上是在学习:")
print("像素变化 与 旋转角度之间的线性关系")

## 预测结果可视化

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(
    y_test_base,
    y_pred_base,
    alpha=0.5
)

min_v = min(y_test_base.min(), y_pred_base.min())
max_v = max(y_test_base.max(), y_pred_base.max())

plt.plot(
    [min_v, max_v],
    [min_v, max_v],
    linestyle="--"
)

plt.xlabel("True Angle")
plt.ylabel("Predicted Angle")

plt.title("传统方法：真实角度 vs 预测角度")

plt.grid(True)

plt.show()

## CNN 回归任务目标

输入：

[batch_size, 1, 28, 28]

即：

每张图像输出一个：

旋转角度预测值

例如：

-13.5°
+28.1°

## 定义 CNN 回归模型

In [ ]:
class RotationCNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(
            1,
            16,
            kernel_size=3,
            padding=1
        )
        
        self.conv2 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )
        
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(
            32 * 7 * 7,
            128
        )
        
        self.fc2 = nn.Linear(
            128,
            1
        )
    
    def forward(self, x):
        
        x = self.pool(
            F.relu(self.conv1(x))
        )
        
        x = self.pool(
            F.relu(self.conv2(x))
        )
        
        x = x.view(x.size(0), -1)
        
        x = F.relu(self.fc1(x))
        
        x = self.fc2(x)
        
        return x.squeeze(1)

model = RotationCNN().to(device)

print(model)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("\n模型参数量:")
print(total_params)

## CNN 张量打印

In [ ]:
class DebugRotationCNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 1)
    
    def forward(self, x):
        
        print("=" * 60)
        
        print("输入张量:")
        print(x.shape)
        
        x = self.conv1(x)
        
        print("\n经过 Conv1 后:")
        print(x.shape)
        print("解释: 1个通道 -> 16个特征图")
        
        x = F.relu(x)
        
        x = self.pool(x)
        
        print("\n经过 Pool1 后:")
        print(x.shape)
        print("解释: 28×28 -> 14×14")
        
        x = self.conv2(x)
        
        print("\n经过 Conv2 后:")
        print(x.shape)
        print("解释: 16个特征图 -> 32个特征图")
        
        x = F.relu(x)
        
        x = self.pool(x)
        
        print("\n经过 Pool2 后:")
        print(x.shape)
        print("解释: 14×14 -> 7×7")
        
        x = x.view(x.size(0), -1)
        
        print("\n展平后:")
        print(x.shape)
        print("解释: 32 × 7 × 7 = 1568")
        
        x = self.fc1(x)
        
        print("\n经过 FC1 后:")
        print(x.shape)
        
        x = F.relu(x)
        
        x = self.fc2(x)
        
        print("\n最终输出:")
        print(x.shape)
        print("解释: 每张图像输出1个角度")
        
        print("=" * 60)
        
        return x.squeeze(1)

debug_model = DebugRotationCNN().to(device)

sample_images, sample_angles = next(iter(train_loader))

sample_images = sample_images.to(device)

with torch.no_grad():
    debug_outputs = debug_model(sample_images)

print("\n真实角度前10个:")
print(sample_angles[:10].numpy())

print("\n预测角度前10个:")
print(debug_outputs[:10].cpu().numpy())

## 训练 CNN

> 💡 **教学提示**：CNN训练是最耗时的环节。观察训练过程中的loss下降趋势，判断模型是否正常收敛。如果loss震荡剧烈，可以尝试降低学习率。

In [ ]:
print("=" * 80)
print("CNN 回归训练配置：Fashion-MNIST 旋转角度预测")
print("=" * 80)

print("任务类型: 图像回归任务")
print("输入张量: [batch_size, 1, 28, 28]")
print("输出张量: [batch_size]")
print("预测目标: 图像旋转角度")
print("损失函数: MSELoss")
print("优化器: Adam")
print("学习率: 0.001")
print("Early Stopping: 连续5个epoch测试MAE不下降则停止")

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 50
patience = 5

best_test_mae = float("inf")
epochs_no_improve = 0
best_model_state = None

train_losses = []
test_maes = []
test_rmses = []
test_r2s = []


def evaluate_regression(model, loader):

    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for images, angles in loader:

            images = images.to(device)
            angles = angles.to(device)

            preds = model(images)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(angles.cpu().numpy())

    preds_all = np.array(preds_all)
    labels_all = np.array(labels_all)

    mae = mean_absolute_error(labels_all, preds_all)

    mse = mean_squared_error(labels_all, preds_all)

    rmse = np.sqrt(mse)

    r2 = r2_score(labels_all, preds_all)

    return mae, rmse, r2


for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    total = 0

    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )

    for images, angles in loop:

        images = images.to(device)
        angles = angles.to(device)

        optimizer.zero_grad()

        preds = model(images)

        loss = criterion(preds, angles)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * angles.size(0)

        total += angles.size(0)

        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / total

    mae, rmse, r2 = evaluate_regression(
        model,
        test_loader
    )

    train_losses.append(epoch_loss)
    test_maes.append(mae)
    test_rmses.append(rmse)
    test_r2s.append(r2)

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
    print(f"Train MSE : {epoch_loss:.4f}")
    print(f"Test MAE  : {mae:.4f}")
    print(f"Test RMSE : {rmse:.4f}")
    print(f"Test R²   : {r2:.4f}")

    # =========================
    # Early Stopping
    # 以 MAE 越小越好作为判断标准
    # =========================

    if mae < best_test_mae:

        best_test_mae = mae
        epochs_no_improve = 0
        best_model_state = model.state_dict()

        print("测试 MAE 下降，保存当前最佳模型。")

    else:

        epochs_no_improve += 1

        print(f"测试 MAE 未下降，累计 {epochs_no_improve} 次。")

    print("-" * 50)

    if epochs_no_improve >= patience:

        print("\n" + "=" * 80)
        print("触发 Early Stopping")
        print(f"连续 {patience} 个 epoch 测试 MAE 未下降")
        print("训练提前结束")
        print("=" * 80)

        break


# =========================
# 恢复最佳模型
# =========================

if best_model_state is not None:

    model.load_state_dict(best_model_state)

    print("\n已恢复最佳模型参数。")


# =========================
# 保存 CNN 回归模型
# =========================

import os

save_dir = "./saved_models"

os.makedirs(
    save_dir,
    exist_ok=True
)

model_save_path = os.path.join(
    save_dir,
    "fashion_mnist_rotation_cnn.pth"
)

checkpoint = {
    "model_state_dict": model.state_dict(),

    "model_name": "RotationCNN",

    "task_type": "image_regression",

    "dataset": "Fashion-MNIST",

    "target": "rotation_angle",

    "angle_range": (-45, 45),

    "input_size": (28, 28),

    "input_channels": 1,

    "loss_function": "MSELoss",

    "best_test_mae": best_test_mae,

    "train_losses": train_losses,

    "test_maes": test_maes,

    "test_rmses": test_rmses,

    "test_r2s": test_r2s,
}

torch.save(
    checkpoint,
    model_save_path
)

print("\n" + "=" * 80)
print("Fashion-MNIST 旋转角预测 CNN 模型已保存")
print("=" * 80)

print("保存路径:", model_save_path)
print("最佳测试 MAE:", best_test_mae)

## 训练可视化

In [ ]:
# =========================
# 使用实际训练轮数
# Early Stopping 后不能再直接用 num_epochs
# =========================

actual_epochs = len(train_losses)

history_df = pd.DataFrame({

    "epoch": range(1, actual_epochs + 1),

    "train_loss": train_losses,

    "test_mae": test_maes,

    "test_rmse": test_rmses,

    "test_r2": test_r2s
})

print("=" * 80)
print("CNN 回归训练历史")
print("=" * 80)

display(history_df)


# =========================
# 训练损失曲线
# =========================

plt.figure(figsize=(7,5))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o"
)

plt.xlabel("Epoch")

plt.ylabel("MSE Loss")

plt.title("CNN训练损失变化")

plt.grid(True)

plt.show()


# =========================
# MAE / RMSE 曲线
# =========================

plt.figure(figsize=(7,5))

plt.plot(
    history_df["epoch"],
    history_df["test_mae"],
    marker="o",
    label="MAE"
)

plt.plot(
    history_df["epoch"],
    history_df["test_rmse"],
    marker="o",
    label="RMSE"
)

plt.legend()

plt.xlabel("Epoch")

plt.ylabel("Error")

plt.title("CNN测试误差变化")

plt.grid(True)

plt.show()


# =========================
# R² 曲线
# =========================

plt.figure(figsize=(7,5))

plt.plot(
    history_df["epoch"],
    history_df["test_r2"],
    marker="o"
)

plt.xlabel("Epoch")

plt.ylabel("R²")

plt.title("CNN回归拟合能力变化")

plt.grid(True)

plt.show()


# =========================
# 课堂解释
# =========================

print("\n课堂解释:")

print("Train Loss:")
print("训练阶段的 MSELoss")

print("\nMAE:")
print("平均绝对误差")
print("越小越好")

print("\nRMSE:")
print("均方根误差")
print("对大误差更敏感")

print("\nR²:")
print("回归拟合优度")
print("越接近1说明拟合越好")

## CNN 最终预测结果

In [ ]:
model.eval()

cnn_preds = []
cnn_labels = []

with torch.no_grad():
    
    for images, angles in test_loader:
        
        images = images.to(device)
        
        preds = model(images)
        
        cnn_preds.extend(
            preds.cpu().numpy()
        )
        
        cnn_labels.extend(
            angles.numpy()
        )

cnn_preds = np.array(cnn_preds)
cnn_labels = np.array(cnn_labels)

cnn_mae = mean_absolute_error(
    cnn_labels,
    cnn_preds
)

cnn_mse = mean_squared_error(
    cnn_labels,
    cnn_preds
)

cnn_rmse = np.sqrt(cnn_mse)

cnn_r2 = r2_score(
    cnn_labels,
    cnn_preds
)

print("CNN MAE:", cnn_mae)
print("CNN RMSE:", cnn_rmse)
print("CNN R²:", cnn_r2)

print("\n前10个真实角度:")
print(cnn_labels[:10])

print("\n前10个预测角度:")
print(cnn_preds[:10])

## 真实值 *vs* 预测值

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(
    cnn_labels,
    cnn_preds,
    alpha=0.5
)

min_v = min(
    cnn_labels.min(),
    cnn_preds.min()
)

max_v = max(
    cnn_labels.max(),
    cnn_preds.max()
)

plt.plot(
    [min_v, max_v],
    [min_v, max_v],
    linestyle="--"
)

plt.xlabel("True Angle")
plt.ylabel("Predicted Angle")

plt.title("CNN：真实角度 vs 预测角度")

plt.grid(True)

plt.show()

## 传统方法 *vs* CNN

In [ ]:
compare_df = pd.DataFrame({
    "方法": [
        "图像展平 + Ridge回归",
        "CNN回归"
    ],
    
    "输入形式": [
        "784维向量",
        "1×28×28图像张量"
    ],
    
    "是否保留空间结构": [
        "否",
        "是"
    ],
    
    "MAE": [
        baseline_mae,
        cnn_mae
    ],
    
    "RMSE": [
        baseline_rmse,
        cnn_rmse
    ],
    
    "R²": [
        baseline_r2,
        cnn_r2
    ]
})

display(compare_df)

plt.figure(figsize=(8,5))

plt.bar(
    compare_df["方法"],
    compare_df["MAE"]
)

plt.ylabel("MAE")

plt.title("传统方法 vs CNN")

plt.show()

# 房价预测任务（House Price Prediction）

## 项目主题

## 房屋图像 + 结构化特征的多模态房价预测

本实验基于：

```text
House Prices and Images - SoCal
```

数据集，利用：

- 房屋图片
- 房屋属性信息

共同预测房价。

---

## 实验目标

通过本实验，理解：

1. 图像回归任务的基本流程；
2. CNN 如何提取房屋视觉特征；
3. 房价预测为什么需要结构化特征；
4. 多模态模型（图像 + 表格）的基本思想；
5. 如何使用 MAE、RMSE、R² 评估回归模型。

### 学习重点

- 记录数据集的基本统计信息
- 为模型设计提供依据

In [ ]:
import os
import random
from pathlib import Path

import shutil

import kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tqdm import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("当前设备:", device)
print("PyTorch版本:", torch.__version__)


plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

## 下载房价数据集

In [ ]:
print("=" * 80)
print("Kaggle 房价数据集下载与项目化管理")
print("=" * 80)

# =========================
# 当前 notebook 所在目录
# =========================

BASE_DIR = Path.cwd()

print("当前工作目录:")
print(BASE_DIR)

# =========================
# teach/data
# =========================

DATA_DIR = BASE_DIR / "data"

DATA_DIR.mkdir(exist_ok=True)

print("\n项目数据目录:")
print(DATA_DIR)

# =========================
# 开始下载
# =========================

print("\n开始下载 Kaggle 数据集...")

dataset_path = kagglehub.dataset_download(
    "ted8080/house-prices-and-images-socal"
)

print("\n原始缓存路径:")
print(dataset_path)

# =========================
# 目标目录
# =========================

src = Path(dataset_path)

dst = DATA_DIR / "house_prices_socal"

# 如果已存在则删除
if dst.exists():

    print("\n检测到旧数据集，先删除...")

    shutil.rmtree(dst)

# =========================
# 移动数据集
# =========================

print("\n正在移动到 teach/data/...")

shutil.move(
    str(src),
    str(dst)
)

print("移动完成。")

# =========================
# 删除 kagglehub 默认缓存
# =========================

cache_dir = Path.home() / ".cache" / "kagglehub"

if cache_dir.exists():

    print("\n删除 kagglehub 默认缓存目录...")

    shutil.rmtree(cache_dir)

    print("缓存已删除。")

else:

    print("\n未检测到 kagglehub 缓存目录。")

# =========================
# 最终目录
# =========================

print("\n" + "=" * 80)
print("最终项目目录结构")
print("=" * 80)

for p in dst.rglob("*"):

    try:
        print(p.relative_to(BASE_DIR))
    except:
        pass

# =========================
# 最终 DATA_DIR
# =========================

DATASET_DIR = dst

print("\n" + "=" * 80)
print("后续统一使用的数据集路径")
print("=" * 80)

print(DATASET_DIR)

print("\n推荐后续路径写法：")

print("""
CSV_PATH = DATASET_DIR / "socal2.csv"

IMAGE_DIR = DATASET_DIR / "socal2" / "socal_pics"
""")

## 读取房价数据结构

In [ ]:
CSV_PATH = DATASET_DIR / "socal2.csv"

IMAGE_DIR = DATASET_DIR / "socal2" / "socal_pics"

df = pd.read_csv(CSV_PATH)

print("=" * 80)
print("房价数据集")
print("=" * 80)

print("DataFrame shape:")
print(df.shape)

print("\n列名:")
print(df.columns.tolist())

print("\n前5行:")
display(df.head())

print("\n价格统计:")
display(df["price"].describe())

## 可视化房屋图片

In [ ]:
plt.figure(figsize=(12,12))

for i in range(16):

    row = df.iloc[i]

    image_id = row["image_id"]

    price = row["price"]

    sqft = row["sqft"]

    bed = row["bed"]

    bath = row["bath"]

    image_path = IMAGE_DIR / f"{image_id}.jpg"

    image = Image.open(image_path)

    plt.subplot(4,4,i+1)

    plt.imshow(image)

    plt.title(
        f"${price:,}\n"
        f"{sqft} sqft\n"
        f"{bed}bd {bath}ba",
        fontsize=9
    )

    plt.axis("off")

plt.tight_layout()

plt.show()

## 观测房价分布

In [ ]:
plt.figure(figsize=(7,5))

plt.hist(
    df["price"],
    bins=50
)

plt.xlabel("Price")

plt.ylabel("Count")

plt.title("原始房价分布")

plt.grid(True)

plt.show()


plt.figure(figsize=(7,5))

plt.hist(
    np.log1p(df["price"]),
    bins=50
)

plt.xlabel("log(price)")

plt.ylabel("Count")

plt.title("log房价分布")

plt.grid(True)

plt.show()


print("=" * 80)
print("课堂解释")
print("=" * 80)

print("房价跨度非常大，因此通常对价格做 log 变换。")

print("\n这样可以：")
print("1. 缓解极端值影响")
print("2. 让回归更稳定")
print("3. 更符合很多真实机器学习任务")

## 构建图像路径

In [ ]:
df["image_path"] = df["image_id"].apply(
    lambda x: str(IMAGE_DIR / f"{x}.jpg")
)

print(df[["image_id", "image_path", "price"]].head())

## 删除缺失图片

In [ ]:
exists_mask = df["image_path"].apply(
    lambda x: Path(x).exists()
)

df = df[exists_mask].reset_index(drop=True)

print("保留后的样本数量:")
print(len(df))

## 设置目标变量

In [ ]:
df["target_price"] = np.log1p(df["price"])

print(df[["price", "target_price"]].head())

## 构建 Dataset 和 DataLoader

In [ ]:
# 图像路径
IMAGE_DIR = DATASET_DIR / "socal2" / "socal_pics"

df["image_path"] = df["image_id"].apply(
    lambda x: str(IMAGE_DIR / f"{x}.jpg")
)

# 删除不存在的图片
df = df[df["image_path"].apply(lambda x: Path(x).exists())].reset_index(drop=True)

# 课堂演示取部分数据
max_samples = min(5000, len(df))
df_small = df.sample(n=max_samples, random_state=42).reset_index(drop=True)

print("用于教学的数据量:", len(df_small))
display(df_small[["image_id", "image_path", "price", "target_price"]].head())


transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])


class HousePriceImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(row["target_price"], dtype=torch.float32)

        return image, target


dataset = HousePriceImageDataset(df_small, transform=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("训练集:", len(train_dataset))
print("测试集:", len(test_dataset))

## 定义简单 CNN 模型

### CNN 结构详解

| 层 | 输入形状 | 输出形状 | 参数量 | 作用 |
|----|---------|---------|--------|------|
| Conv2d(3→32,3×3) | [B,3,32,32] | [B,32,32,32] | 896 | 提取低阶特征（边缘、颜色） |
| ReLU | [B,32,32,32] | [B,32,32,32] | 0 | 引入非线性 |
| MaxPool2d(2×2) | [B,32,32,32] | [B,32,16,16] | 0 | 降采样，提取主要特征 |
| Conv2d(32→64,3×3) | [B,32,16,16] | [B,64,16,16] | 18496 | 提取中阶特征（纹理、形状） |
| ReLU | ... | ... | 0 | 引入非线性 |
| MaxPool2d(2×2) | [B,64,16,16] | [B,64,8,8] | 0 | 进一步降采样 |
| Flatten | [B,64,8,8] | [B,4096] | 0 | 展开成一维 |
| Linear(4096→256) | [B,4096] | [B,256] | 1,048,832 | 全连接特征整合 |
| Linear(256→10) | [B,256] | [B,10] | 2,570 | 输出10类得分 |

> 💡 **教学提示**：可以把CNN想象成"漏斗"——从大尺寸、少通道的图像开始，逐步变成小尺寸、多通道的特征图，最后展开成一维向量进行分类。

In [ ]:
class HousePriceCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # 128 -> 64 -> 32 -> 16
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [B, 16, 64, 64]
        x = self.pool(F.relu(self.conv2(x)))  # [B, 32, 32, 32]
        x = self.pool(F.relu(self.conv3(x)))  # [B, 64, 16, 16]

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x.squeeze(1)


device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = HousePriceCNN().to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters())
print("模型总参数量:", total_params)

## CNN 的明确任务目标

In [ ]:
print("=" * 80)
print("CNN 房价预测任务目标")
print("=" * 80)

print("任务类型: 图像回归任务")
print("输入: 房屋图片")
print("输入张量: [batch_size, 3, 128, 128]")
print("输出: 房价的 log(price)")
print("输出张量: [batch_size]")
print("损失函数: MSELoss")
print("评价指标: MAE / RMSE / R²")

print("\n为什么预测 log(price):")
print("房价跨度很大，直接预测原始价格不稳定。")
print("使用 log(price) 可以缓解极端高价样本对模型训练的影响。")

print("\n课堂提醒:")
print("房价不只由图片决定，还受地段、面积、房龄、城市等因素影响。")
print("所以本任务用于演示 CNN 图像回归，而不是严格商业估价。")

## 用一个 batch 检查 CNN 前向传播

### 张量形状追踪

检查前向传播时，关注每个层的输入输出形状是否匹配：
```
输入图像: [B, 3, 32, 32]  ← CIFAR-10 的32×32彩色图
  ↓ Conv+Pool
特征图:   [B, 64, 8, 8]    ← 经过两次卷积+池化
  ↓ Flatten
一维向量: [B, 4096]         ← 展开
  ↓ Linear+Linear
类别得分: [B, 10]           ← 10个类别的logits
```

### 📚 学习重点

- 理解深度学习中的batch概念（一次性送入模型的样本数）
- 掌握查看张量形状的方法，这是调试模型的必备技能
- 图像数据形状为 [B, C, H, W]：B=batch, C=通道, H=高, W=宽

In [ ]:
images, targets = next(iter(train_loader))

images = images.to(device)
targets = targets.to(device)

with torch.no_grad():
    outputs = model(images)

print("=" * 80)
print("一个 batch 的 CNN 前向传播检查")
print("=" * 80)

print("输入图像 batch shape:", images.shape)
print("真实标签 batch shape:", targets.shape)
print("模型输出 shape:", outputs.shape)

print("\n前5个真实 log(price):")
print(targets[:5].detach().cpu().numpy())

print("\n前5个预测 log(price):")
print(outputs[:5].detach().cpu().numpy())

print("\n换算为原始价格：")
true_prices = np.expm1(targets[:5].detach().cpu().numpy())
pred_prices = np.expm1(outputs[:5].detach().cpu().numpy())

preview_df = pd.DataFrame({
    "真实价格": true_prices,
    "预测价格": pred_prices,
    "绝对误差": np.abs(true_prices - pred_prices)
})

display(preview_df)

print("\n说明:")
print("模型训练前参数是随机初始化的，因此预测结果通常不可靠。")

## 训练 CNN 模型

> 💡 **教学提示**：CNN训练比全连接网络慢很多，因为卷积层需要大量计算。观察训练过程中损失是否持续下降，准确率是否持续上升。如果训练损失下降但验证损失上升，说明发生了过拟合。

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50
patience = 5

best_test_mae = float("inf")
epochs_no_improve = 0
best_model_state = None

train_losses = []
test_maes = []
test_rmses = []
test_r2s = []


def evaluate_regression(model, loader):
    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(targets.cpu().numpy())

    preds_all = np.array(preds_all)
    labels_all = np.array(labels_all)

    mae_log = mean_absolute_error(labels_all, preds_all)
    mse_log = mean_squared_error(labels_all, preds_all)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(labels_all, preds_all)

    return mae_log, rmse_log, r2_log


print("=" * 80)
print("开始训练 CNN 房价预测模型")
print("=" * 80)

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")

    for images, targets in loop:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        preds = model(images)
        loss = criterion(preds, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * targets.size(0)
        total += targets.size(0)

        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / total

    test_mae, test_rmse, test_r2 = evaluate_regression(model, test_loader)

    train_losses.append(epoch_loss)
    test_maes.append(test_mae)
    test_rmses.append(test_rmse)
    test_r2s.append(test_r2)

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
    print(f"Train MSE Loss: {epoch_loss:.4f}")
    print(f"Test MAE  on log(price): {test_mae:.4f}")
    print(f"Test RMSE on log(price): {test_rmse:.4f}")
    print(f"Test R²   on log(price): {test_r2:.4f}")

    if test_mae < best_test_mae:
        best_test_mae = test_mae
        epochs_no_improve = 0
        best_model_state = model.state_dict()
        print("测试 MAE 下降，保存当前最佳模型。")
    else:
        epochs_no_improve += 1
        print(f"测试 MAE 未下降，累计 {epochs_no_improve} 次。")

    print("-" * 50)

    if epochs_no_improve >= patience:
        print("触发 Early Stopping，训练提前结束。")
        break


if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("已恢复最佳模型参数。")


save_dir = "./saved_models_house"
os.makedirs(save_dir, exist_ok=True)

model_save_path = os.path.join(save_dir, "house_price_cnn.pth")

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_name": "HousePriceCNN",
    "task_type": "image_regression",
    "target": "log_price",
    "input_size": (128, 128),
    "mean": (0.5, 0.5, 0.5),
    "std": (0.5, 0.5, 0.5),
    "best_test_mae_log": best_test_mae,
    "train_losses": train_losses,
    "test_maes": test_maes,
    "test_rmses": test_rmses,
    "test_r2s": test_r2s,
}

torch.save(checkpoint, model_save_path)

print("模型已保存:", model_save_path)

## 为什么要构建结构化特征（Structured Features）

## 一、单纯使用房屋图片的问题

在前面的实验中，我们尝试仅使用房屋外观图片预测房价：

```text
房屋图片 → CNN → 房价
```

虽然 CNN 可以学习：

- 建筑外观
- 装修风格
- 房屋大小的视觉印象
- 周围环境

但模型效果并不好，甚至测试集 R² 出现负值。

这说明：

> 房价并不是一个“纯视觉变量”。

---

## 二、现实中的房价由哪些因素决定？

现实中的房价通常同时受到：

| 类型 | 例子 |
|---|---|
| 视觉信息 | 外观、装修、环境 |
| 房屋属性 | 面积、卧室数、浴室数 |
| 地理信息 | 城市、地段、学区 |
| 市场信息 | 房地产市场行情 |

影响。

而 CNN 只能看到：

```text
房屋图片
```

却无法直接知道：

- 房屋面积是多少
- 有几个卧室
- 位于哪个城市
- 属于什么地段

因此：

> 单纯依赖图片进行房价预测，本身存在信息缺失。

---

## 三、解决方案：多模态学习（Multimodal Learning）

因此，我们引入：

## 结构化特征（Structured Features）

例如：

```python
[
    "bed",
    "bath",
    "sqft",
    "n_citi"
]
```

这些特征本质上是：

```text
表格数据（Tabular Data）
```

于是模型结构变为：

```text
房屋图片
    ↓
CNN提取视觉特征

+
    
结构化表格特征
    ↓
MLP提取数值特征

↓

特征融合（Fusion）

↓

预测房价
```

---

## 四、这种方法为什么更合理？

因为：

## CNN 擅长：

- 图像
- 纹理
- 外观
- 空间结构

## MLP（全连接网络）擅长：

- 数值变量
- 面积
- 房间数量
- 城市编码

因此：

> 多模态模型能够同时利用视觉信息与非视觉信息。

这比：

```text
只看图片
```

更加接近真实工业系统。

---

## 五、这也是很多真实 AI 系统的做法

现实中的 AI 系统通常并不是：

```text
只看图片
```

而是：

```text
图像 + 表格 + 文本 + 传感器数据
```

共同决策。

例如：

| 场景 | 多模态信息 |
|---|---|
| 自动驾驶 | 摄像头 + 雷达 + GPS |
| 医疗AI | CT图像 + 病历 |
| 推荐系统 | 用户行为 + 商品图片 |
| 房价预测 | 房屋图片 + 房屋属性 |

---

## 六、本实验的教学意义

本实验不仅是在训练 CNN。

更重要的是：

> 理解“模型需要足够的信息才能完成任务”。

即：

```text
AI 并不是万能的。
如果输入信息不完整，
模型性能就会受到限制。
```

因此：

## 从单模态（图像）
## 升级到多模态（图像 + 表格）

是本实验的重要目标。

### 为什么需要结构化特征？

房价预测不仅仅取决于"房子长什么样"（图像特征），还取决于：
- 卧室数量、面积、楼层等结构化信息
- 这些信息无法从图像中直接获取

因此我们构建**多模态模型**——同时处理图像特征和结构化特征。这更接近真实世界中的AI应用。

> 💡 **教学提示**：CNN训练是最耗时的环节。观察训练过程中的loss下降趋势，判断模型是否正常收敛。如果loss震荡剧烈，可以尝试降低学习率。

## 构建结构化特征

### 为什么需要结构化特征？

房价预测不仅仅取决于"房子长什么样"（图像特征），还取决于：
- 卧室数量、面积、楼层等结构化信息
- 这些信息无法从图像中直接获取

因此我们构建**多模态模型**——同时处理图像特征和结构化特征。这更接近真实世界中的AI应用。

In [ ]:
feature_cols = [
    "bed",
    "bath",
    "sqft",
    "n_citi"
]

print("使用的结构化特征:")
print(feature_cols)

X_struct = df_small[feature_cols]

print("\n前5行:")
display(X_struct.head())

print("\n统计信息:")
display(X_struct.describe())

## 标准化结构化特征

In [ ]:
scaler = StandardScaler()

df_small[feature_cols] = scaler.fit_transform(
    df_small[feature_cols]
)

print("标准化后的前5行:")
display(df_small[feature_cols].head())

## 构建多模态 Dataset

In [ ]:
class MultiModalHouseDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # =========================
        # 图像
        # =========================

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # =========================
        # 结构化特征
        # =========================

        structured_features = torch.tensor(
            row[feature_cols].values.astype(np.float32)
        )

        # =========================
        # 标签
        # =========================

        target = torch.tensor(
            row["target_price"],
            dtype=torch.float32
        )

        return image, structured_features, target

## 重建训练 Dataset

In [ ]:
dataset = MultiModalHouseDataset(
    df_small,
    transform=transform
)

train_size = int(0.8 * len(dataset))

test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("训练集:", len(train_dataset))
print("测试集:", len(test_dataset))

## 检查 batch

In [ ]:
images, structured_features, targets = next(
    iter(train_loader)
)

print("=" * 80)
print("多模态 batch")
print("=" * 80)

print("图像 shape:")
print(images.shape)

print("\n结构化特征 shape:")
print(structured_features.shape)

print("\n标签 shape:")
print(targets.shape)

print("\n前5个结构化特征:")
print(structured_features[:5])

print("\n前5个 log(price):")
print(targets[:5])

## 多模态 CNN 回归模型

### 多模态架构

```
图像输入 (224×224×3)          结构化输入 (7维)
    ↓                               ↓
  CNN编码器                   Linear层
    ↓                               ↓
  图像特征向量 (512维)         结构化特征向量 (64维)
    ↓                               ↓
             拼接 (512+64=576维)
                    ↓
              全连接层
                    ↓
              房价预测 (1维)
```

> 💡 **教学提示**：多模态融合是AI的前沿方向。人类理解世界也是通过视觉、听觉、触觉等多种感官的综合。

In [ ]:
class MultiModalHouseCNN(nn.Module):

    def __init__(self):

        super().__init__()

        # =========================
        # CNN 图像分支
        # =========================

        self.conv1 = nn.Conv2d(
            3,
            16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            32,
            64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2,2)

        # 128 -> 64 -> 32 -> 16
        self.image_fc = nn.Linear(
            64 * 16 * 16,
            128
        )

        # =========================
        # 结构化特征分支
        # =========================

        self.struct_fc = nn.Linear(
            len(feature_cols),
            32
        )

        # =========================
        # 融合层
        # =========================

        self.fusion_fc1 = nn.Linear(
            128 + 32,
            64
        )

        self.fusion_fc2 = nn.Linear(
            64,
            1
        )

    def forward(
        self,
        image,
        structured_features
    ):

        # =========================
        # CNN分支
        # =========================

        x_img = self.pool(
            F.relu(self.conv1(image))
        )

        x_img = self.pool(
            F.relu(self.conv2(x_img))
        )

        x_img = self.pool(
            F.relu(self.conv3(x_img))
        )

        x_img = x_img.view(
            x_img.size(0),
            -1
        )

        x_img = F.relu(
            self.image_fc(x_img)
        )

        # =========================
        # 表格分支
        # =========================

        x_struct = F.relu(
            self.struct_fc(structured_features)
        )

        # =========================
        # 特征融合
        # =========================

        x = torch.cat(
            [x_img, x_struct],
            dim=1
        )

        x = F.relu(
            self.fusion_fc1(x)
        )

        x = self.fusion_fc2(x)

        return x.squeeze(1)

In [ ]:
# =========================
# 初始化模型
# =========================

model = MultiModalHouseCNN().to(device)

print(model)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n模型参数量:")
print(total_params)

## 检查前向传播

In [ ]:
images, structured_features, targets = next(
    iter(train_loader)
)

images = images.to(device)

structured_features = structured_features.to(device)

with torch.no_grad():

    outputs = model(
        images,
        structured_features
    )

print("输出 shape:")
print(outputs.shape)

print("\n前5个预测:")
print(outputs[:5].cpu().numpy())

print("\n前5个真实值:")
print(targets[:5].numpy())

## 修改 evaluate_regression

In [ ]:
def evaluate_regression(model, loader):

    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for images, structured_features, targets in loader:

            images = images.to(device)

            structured_features = structured_features.to(device)

            targets = targets.to(device)

            preds = model(
                images,
                structured_features
            )

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                targets.cpu().numpy()
            )

    preds_all = np.array(preds_all)

    labels_all = np.array(labels_all)

    mae = mean_absolute_error(
        labels_all,
        preds_all
    )

    mse = mean_squared_error(
        labels_all,
        preds_all
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        labels_all,
        preds_all
    )

    return mae, rmse, r2

## 多模态 CNN 训练循环

> 💡 **教学提示**：CNN训练是最耗时的环节。观察训练过程中的loss下降趋势，判断模型是否正常收敛。如果loss震荡剧烈，可以尝试降低学习率。

In [ ]:
# =========================
# 多模态 CNN 训练循环
# 图像 + 结构化特征 → log(price)
# 1. 使用 copy.deepcopy 保存最佳模型
# 2. Early Stopping 后重新评估恢复后的最佳模型
# 3. 保存 checkpoint
# =========================

import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


print("=" * 80)
print("多模态 CNN 房价预测模型训练：修正版")
print("=" * 80)

print("任务类型: 多模态回归")
print("图像输入: [batch_size, 3, 128, 128]")
print("结构化输入:", feature_cols)
print("输出目标: log(price)")
print("损失函数: MSELoss")
print("优化器: Adam")
print("学习率: 0.001")
print("早停机制: 连续5个 epoch 测试 MAE 不下降则停止")

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 50
patience = 5

best_test_mae = float("inf")
best_epoch = 0
epochs_no_improve = 0
best_model_state = None

train_losses = []
test_maes = []
test_rmses = []
test_r2s = []


# =========================
# 评估函数
# =========================

def evaluate_regression(model, loader):

    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for images, structured_features, targets in loader:

            images = images.to(device)
            structured_features = structured_features.to(device)
            targets = targets.to(device)

            preds = model(
                images,
                structured_features
            )

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                targets.cpu().numpy()
            )

    preds_all = np.array(preds_all)
    labels_all = np.array(labels_all)

    mae = mean_absolute_error(
        labels_all,
        preds_all
    )

    mse = mean_squared_error(
        labels_all,
        preds_all
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        labels_all,
        preds_all
    )

    return mae, rmse, r2, preds_all, labels_all


# =========================
# 开始训练
# =========================

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    total = 0

    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )

    for images, structured_features, targets in loop:

        images = images.to(device)
        structured_features = structured_features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        preds = model(
            images,
            structured_features
        )

        loss = criterion(
            preds,
            targets
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * targets.size(0)
        total += targets.size(0)

        loop.set_postfix(
            loss=loss.item()
        )

    epoch_loss = running_loss / total

    test_mae, test_rmse, test_r2, _, _ = evaluate_regression(
        model,
        test_loader
    )

    train_losses.append(epoch_loss)
    test_maes.append(test_mae)
    test_rmses.append(test_rmse)
    test_r2s.append(test_r2)

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
    print(f"Train MSE Loss          : {epoch_loss:.4f}")
    print(f"Test MAE  on log(price) : {test_mae:.4f}")
    print(f"Test RMSE on log(price) : {test_rmse:.4f}")
    print(f"Test R²   on log(price) : {test_r2:.4f}")

    # =========================
    # Early Stopping
    # =========================

    if test_mae < best_test_mae:

        best_test_mae = test_mae
        best_epoch = epoch + 1
        epochs_no_improve = 0

        # 关键：深拷贝，防止后续训练覆盖最佳模型
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        print("测试 MAE 下降，保存当前最佳模型。")

    else:

        epochs_no_improve += 1

        print(f"测试 MAE 未下降，累计 {epochs_no_improve} 次。")

    print("-" * 50)

    if epochs_no_improve >= patience:

        print("\n" + "=" * 80)
        print("触发 Early Stopping")
        print(f"连续 {patience} 个 epoch 测试 MAE 未下降")
        print("训练提前结束")
        print("=" * 80)

        break


# =========================
# 恢复最佳模型
# =========================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    print("\n已恢复最佳模型参数。")
    print("最佳模型来自 epoch:", best_epoch)
    print("最佳测试 MAE on log(price):", best_test_mae)


# =========================
# 重新评估最佳模型
# =========================

final_mae, final_rmse, final_r2, final_preds_log, final_labels_log = evaluate_regression(
    model,
    test_loader
)

print("\n" + "=" * 80)
print("恢复后的最佳模型性能")
print("=" * 80)

print(f"Final MAE  on log(price): {final_mae:.4f}")
print(f"Final RMSE on log(price): {final_rmse:.4f}")
print(f"Final R²   on log(price): {final_r2:.4f}")


# =========================
# 换算回原始房价尺度
# =========================

final_preds_price = np.expm1(final_preds_log)
final_labels_price = np.expm1(final_labels_log)

final_price_mae = mean_absolute_error(
    final_labels_price,
    final_preds_price
)

final_price_mse = mean_squared_error(
    final_labels_price,
    final_preds_price
)

final_price_rmse = np.sqrt(
    final_price_mse
)

final_price_r2 = r2_score(
    final_labels_price,
    final_preds_price
)

print("\n" + "=" * 80)
print("恢复后的最佳模型性能：原始房价尺度")
print("=" * 80)

print(f"Final MAE  on price: ${final_price_mae:,.2f}")
print(f"Final RMSE on price: ${final_price_rmse:,.2f}")
print(f"Final R²   on price: {final_price_r2:.4f}")


# =========================
# 保存模型
# =========================

save_dir = "./saved_models_house"

os.makedirs(
    save_dir,
    exist_ok=True
)

model_save_path = os.path.join(
    save_dir,
    "house_price_multimodal_cnn.pth"
)

checkpoint = {

    "model_state_dict": model.state_dict(),

    "model_name": "MultiModalHouseCNN",

    "task_type": "multimodal_regression",

    "target": "log_price",

    "feature_cols": feature_cols,

    "input_size": (128, 128),

    "mean": (0.5, 0.5, 0.5),

    "std": (0.5, 0.5, 0.5),

    "best_epoch": best_epoch,

    "best_test_mae_log": best_test_mae,

    "final_mae_log": final_mae,

    "final_rmse_log": final_rmse,

    "final_r2_log": final_r2,

    "final_mae_price": final_price_mae,

    "final_rmse_price": final_price_rmse,

    "final_r2_price": final_price_r2,

    "train_losses": train_losses,

    "test_maes": test_maes,

    "test_rmses": test_rmses,

    "test_r2s": test_r2s,
}

torch.save(
    checkpoint,
    model_save_path
)

print("\n" + "=" * 80)
print("多模态 CNN 模型已保存")
print("=" * 80)

print("保存路径:", model_save_path)
print("最佳 epoch:", best_epoch)
print("最佳测试 MAE on log(price):", best_test_mae)

## 多模态模型训练过程可视化

In [ ]:
actual_epochs = len(train_losses)

history_df = pd.DataFrame({
    "epoch": range(1, actual_epochs + 1),
    "train_loss": train_losses,
    "test_mae_log": test_maes,
    "test_rmse_log": test_rmses,
    "test_r2_log": test_r2s
})

print("=" * 80)
print("训练历史")
print("=" * 80)

display(history_df)

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss on log(price)")
plt.title("训练损失变化")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["test_mae_log"], marker="o", label="MAE")
plt.plot(history_df["epoch"], history_df["test_rmse_log"], marker="o", label="RMSE")
plt.xlabel("Epoch")
plt.ylabel("Error on log(price)")
plt.title("测试误差变化")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(history_df["epoch"], history_df["test_r2_log"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("R² on log(price)")
plt.title("测试集 R² 变化")
plt.grid(True)
plt.show()

## 测试集整体预测结果

### 学习重点

- 评估模型在**未见过的数据**上的表现
- 测试集准确率是衡量模型泛化能力的最终标准

In [ ]:
model.eval()

all_preds_log = []
all_labels_log = []
all_images = []
all_structured = []

with torch.no_grad():
    for images, structured_features, targets in test_loader:
        images_device = images.to(device)
        structured_device = structured_features.to(device)

        preds = model(images_device, structured_device)

        all_preds_log.extend(preds.cpu().numpy())
        all_labels_log.extend(targets.numpy())
        all_images.extend(images.cpu())
        all_structured.extend(structured_features.cpu().numpy())

all_preds_log = np.array(all_preds_log)
all_labels_log = np.array(all_labels_log)

all_preds_price = np.expm1(all_preds_log)
all_labels_price = np.expm1(all_labels_log)

errors_price = np.abs(all_labels_price - all_preds_price)

mae_log = mean_absolute_error(all_labels_log, all_preds_log)
rmse_log = np.sqrt(mean_squared_error(all_labels_log, all_preds_log))
r2_log = r2_score(all_labels_log, all_preds_log)

mae_price = mean_absolute_error(all_labels_price, all_preds_price)
rmse_price = np.sqrt(mean_squared_error(all_labels_price, all_preds_price))
r2_price = r2_score(all_labels_price, all_preds_price)

print("=" * 80)
print("测试集整体结果")
print("=" * 80)

print("log(price) 尺度:")
print("MAE :", mae_log)
print("RMSE:", rmse_log)
print("R²  :", r2_log)

print("\n原始房价尺度:")
print(f"MAE : ${mae_price:,.2f}")
print(f"RMSE: ${rmse_price:,.2f}")
print(f"R²  : {r2_price:.4f}")

result_df = pd.DataFrame({
    "真实价格": all_labels_price,
    "预测价格": all_preds_price,
    "绝对误差": errors_price,
    "真实log价格": all_labels_log,
    "预测log价格": all_preds_log
})

display(result_df.head(10))

In [ ]:
# =========================
# 真实价格 vs 预测价格
# =========================

plt.figure(figsize=(6, 6))

plt.scatter(
    all_labels_price,
    all_preds_price,
    alpha=0.4
)

min_v = min(all_labels_price.min(), all_preds_price.min())
max_v = max(all_labels_price.max(), all_preds_price.max())

plt.plot(
    [min_v, max_v],
    [min_v, max_v],
    linestyle="--"
)

plt.xlabel("True Price")
plt.ylabel("Predicted Price")
plt.title("真实房价 vs 预测房价")
plt.grid(True)
plt.show()


# log尺度也画一张
plt.figure(figsize=(6, 6))

plt.scatter(
    all_labels_log,
    all_preds_log,
    alpha=0.4
)

min_v = min(all_labels_log.min(), all_preds_log.min())
max_v = max(all_labels_log.max(), all_preds_log.max())

plt.plot(
    [min_v, max_v],
    [min_v, max_v],
    linestyle="--"
)

plt.xlabel("True log(price)")
plt.ylabel("Predicted log(price)")
plt.title("真实 log(price) vs 预测 log(price)")
plt.grid(True)
plt.show()

## 误差分布

### 学习重点

- 误差分布图展示了预测误差的分布情况
- 理想情况：误差以0为中心呈正态分布
- 如果误差有偏斜，说明模型存在系统性偏差

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(errors_price, bins=40)
plt.xlabel("Absolute Error in Price")
plt.ylabel("Count")
plt.title("原始房价绝对误差分布")
plt.grid(True)
plt.show()

errors_log = np.abs(all_labels_log - all_preds_log)

plt.figure(figsize=(7, 5))
plt.hist(errors_log, bins=40)
plt.xlabel("Absolute Error in log(price)")
plt.ylabel("Count")
plt.title("log(price)绝对误差分布")
plt.grid(True)
plt.show()

print("=" * 80)
print("误差统计")
print("=" * 80)

print(f"原始房价平均绝对误差: ${errors_price.mean():,.2f}")
print(f"原始房价中位数绝对误差: ${np.median(errors_price):,.2f}")
print(f"原始房价最大绝对误差: ${errors_price.max():,.2f}")

print("\nlog(price) 平均绝对误差:", errors_log.mean())
print("log(price) 中位数绝对误差:", np.median(errors_log))
print("log(price) 最大绝对误差:", errors_log.max())